In [1]:
import heapq
import random
from bisect import bisect_left, bisect_right
from collections import Counter, defaultdict, deque
from functools import cache, cmp_to_key

# LeetCode Playbook

---
# 1) Complexity Budget

Python does roughly $10^7$ simple operations a second (compiled languages ~$10^8$). So the input size tells you the complexity they expect — often the biggest hint in the whole statement: n ≤ 20 → exponential is fine, n ≤ 10³ → O(n²), n ≤ 10⁵ → O(n log n) or O(n), huge values → O(log n).

**Cost of common Python operations**

| Operation | Cost | Note |
|---|---|---|
| `list.append`, `list.pop()` | O(1) | end of list |
| `list.pop(0)`, `list.insert(0, x)`, `x in list` | O(n) | use `deque` / `set` instead |
| `deque.appendleft`, `deque.popleft` | O(1) | queues and BFS |
| `dict` / `set` lookup, insert, delete | O(1) average | keys must be hashable (tuples, not lists) |
| `heapq.heappush`, `heappop` | O(log n) | min-heap; negate for max-heap |
| `heapq.heapify` | O(n) | faster than n pushes |
| `bisect_left` / `bisect_right` | O(log n) | on a sorted list |
| `sorted`, `list.sort` | O(n log n) | stable; `key=` for custom order |
| `s[i:j]`, `s + t` | O(length) | slicing copies |
| `"".join(parts)` | O(total) | build strings with a list, not `+=` in a loop |

---
# 2) Python Toolkit

The standard library already has most data structures you need. These are the calls the practice notebooks use.

In [2]:
# Counter: frequencies, multiset arithmetic, top-k
count = Counter("mississippi")
print(count.most_common(2), Counter("aab") - Counter("ab"))

# defaultdict: grouping without key checks
groups = defaultdict(list)
for word in ["eat", "tea", "tan"]:
    groups["".join(sorted(word))].append(word)
print(dict(groups))

# deque: O(1) at both ends for queues and sliding windows
dq = deque([1, 2, 3])
dq.appendleft(0)
dq.pop()
print(dq)

[('i', 4), ('s', 4)] Counter({'a': 1})
{'aet': ['eat', 'tea'], 'ant': ['tan']}
deque([0, 1, 2])


In [3]:
# heapq: min-heap; negate values for a max-heap; tuples compare element-wise
heap = [5, 1, 8, 3]
heapq.heapify(heap)
print(heapq.heappop(heap), heapq.nsmallest(2, heap), heapq.nlargest(1, heap))
max_heap = [-x for x in [5, 1, 8]]
heapq.heapify(max_heap)
print(-max_heap[0])

# bisect: insertion points in a sorted list
sorted_nums = [1, 3, 3, 3, 7]
print(bisect_left(sorted_nums, 3), bisect_right(sorted_nums, 3))


# cache: turns a recursive function into top-down DP
@cache
def ways(n):
    """Stair-climbing ways with 1- or 2-steps."""
    return 1 if n <= 1 else ways(n - 1) + ways(n - 2)


print(ways(50))

1 [3, 5] [8]
8
1 4
20365011074


In [4]:
# sorting with keys: by length, then alphabetically; descending numbers
words = ["bb", "a", "ccc", "ab"]
print(sorted(words, key=lambda w: (len(w), w)))
print(sorted([3, 1, 2], reverse=True))

# grid neighbours and bounds checks
DIRS = [(1, 0), (-1, 0), (0, 1), (0, -1)]
rows, cols, r, c = 3, 3, 0, 0
print(
    [(r + dr, c + dc) for dr, dc in DIRS if 0 <= r + dr < rows and 0 <= c + dc < cols]
)

# infinities, integer division and modulo with negatives
print(float("inf") > 10**18, -7 // 2, int(-7 / 2), -7 % 3)

['a', 'ab', 'bb', 'ccc']
[3, 2, 1]
[(1, 0), (0, 1)]
True -4 -3 2


**Python gotchas that cause wrong answers**

| Gotcha | Fix |
|---|---|
| `[[0] * n] * m` shares one row m times | `[[0] * n for _ in range(m)]` |
| `-7 // 2 == -4` (floor), problems often want truncation | `int(a / b)` |
| default recursion limit 1000 | iterative DFS / BFS, or `sys.setrecursionlimit` |
| mutating a list while iterating | iterate over a copy or build a new list |
| `path` appended by reference in backtracking | `out.append(path.copy())` |
| mutable default argument `def f(x=[])` | `def f(x=None)` |
| `heapq` compares the second tuple element on ties | add a counter/index as a tie-breaker |

---
# 3) Pattern Recognition

Left column = what the problem *says*. Right column = what to try first.

| If the problem says… | Try | Practice |
|---|---|---|
| "pair / triple with sum", "seen before", "group by" | hash map / set | 02 §1 |
| sorted array, "pair", "in place", palindrome | two pointers | 02 §2 |
| "longest / shortest substring or subarray with …" | sliding window | 02 §3 |
| "subarray sum equals / divisible by k", negatives allowed | prefix sum + hash map | 02 §4 |
| range sums **and** point updates mixed | Fenwick tree | 02 §4.2 |
| brackets, nested structure, "previous / next greater" | stack / monotonic stack | 03 §1 |
| sorted or rotated input, "minimum X such that feasible" | binary search (on the answer) | 03 §2 |
| linked list: middle, cycle, k-th from end, reverse | slow/fast pointers, dummy head | 03 §3 |
| tree: depth, path, ancestor, balanced | DFS returning values upward | 04 §1 |
| tree: level, right view, nearest | BFS by level | 04 §2 |
| BST, "k-th smallest", sorted order | in-order traversal / bounds | 04 §3 |
| prefixes, autocomplete, many words in a grid | trie | 04 §4 |
| "top k", "k-th largest", streaming median, merge k | heap | 04 §5 |
| grid regions, islands, flood fill | DFS / BFS on grid | 05 §1 |
| "minimum steps", unweighted shortest path | BFS (multi-source if many starts) | 05 §1–2 |
| prerequisites, dependency order | topological sort | 05 §3 |
| "connected?", merging groups, redundant edge | union-find | 05 §4 |
| weighted shortest path, "cheapest", "minimum effort" | Dijkstra / Bellman–Ford | 05 §5 |
| "all combinations / permutations / subsets", n ≤ 20 | backtracking | 05 §6 |
| "number of ways", "min/max cost", overlapping choices | dynamic programming | 06 |
| two strings compared character by character | 2D DP over prefixes | 06 §5 |
| intervals: merge, overlap, rooms | sort by start / end, heap of ends | 07 §2 |
| "minimum number of …" with a clear local choice | greedy (prove it!) | 07 §1 |
| "without extra space", appears twice/once | XOR / bit tricks | 07 §4 |
| random item from a stream / unknown length | reservoir sampling | 07 §3.2 |
| "design a class with O(1) …" | hash map + list / deque / heap combination | 07 §5 |

**When they push further (follow-ups)**

After you solve it, expect a twist. Most follow-ups are one of these:

| They ask… | Usual answer |
|---|---|
| "Can you do it in O(1) extra space?" | reuse the input (sign marks, first row/column as flags, write pointer), or two pointers instead of a map |
| "Faster?" | look for the bound you're not using: sorted → binary search, small values → counting/buckets, heap → quickselect |
| "Data is a stream / arrives one by one" | keep a running summary: heap of size k, running counts, union–find, monotonic deque |
| "Doesn't fit in memory / many machines" | split into chunks, solve each, merge (external sort, map-reduce, top-k per shard) |
| "Return the actual answer, not just the number" | store a parent / choice per state and walk back |
| "Duplicates / negatives / Unicode allowed?" | sort + skip equal siblings · prefix sums instead of a window · dict instead of a 26-array |
| "Many queries on the same data" | precompute once (prefix sums, sorted index, trie), then answer each query fast |
| "Make it thread-safe" | one lock around operations that touch shared state; shard by key to scale |

Each Medium in the practice notebooks ends with its own **Likely follow-ups**.

---
# 4) Pattern Templates

Many questions are the same skeleton with a few lines swapped. Each subtopic of the practice notebooks gets a rating:

| Rating | Meaning | How to study it |
|---|---|---|
| 🟢 Green | one template; each question changes **1–3 lines** | memorise the template, then learn the per-question diff table |
| 🟡 Yellow | 2–3 template shapes; a **condition** picks the shape | learn the condition first ("reads or updates?", "substring or subsequence?") |
| 🔴 Red | no consistent template | learn the handful of ideas; each is a small function |

Inside the code, every line you edit per question sits under a `# CHANGE:` comment.

**Template map** (28 🟢 · 14 🟡 · 10 🔴)

| § | Subtopic | Practice | Rating | Master template | What you change |
|---|---|---|---|---|---|
| 4.1.1 | Hash map and set lookups | 02 §1.1 | 🟡 | seen-before · count-and-compare · group-by-key | which of the 3 shapes, then the key / partner |
| 4.1.2 | In-place array manipulation | 02 §1.2 | 🟡 | write pointer · 3-way partition · reverse pieces | the keep rule, the pivot, or which ranges to reverse |
| 4.1.3 | Opposite-end pointers | 02 §2.1 | 🟢 | `while left < right`: look at both ends, move one | what you compute, when you stop, which side moves |
| 4.1.4 | Same-direction pointers | 02 §2.2 | 🟢 | `i` walks one sequence, `j` follows / reads ahead | when the second pointer moves |
| 4.1.5 | Variable-size windows | 02 §3.1 | 🟢 | extend right → shrink while broken → record | window state, 'broken' condition, max vs min |
| 4.1.6 | Fixed-size windows | 02 §3.2 | 🟢 | add `right`, drop `right - k`, check once full | the state (Counter / sum / deque) and the check |
| 4.1.7 | Prefix sum with hash map | 02 §4.1 | 🟢 | running prefix + map of earlier prefixes | how x moves the prefix, the key, count vs first index |
| 4.1.8 | Range queries and difference arrays | 02 §4.2 | 🟡 | prefix (reads) · difference (updates) · Fenwick (both) | reads, updates or both mixed; 1D vs 2D |
| 4.1.9 | Matrix traversal and transforms | 02 §5.1 | 🔴 | none — four tricks | — |
| 4.1.10 | String parsing and simulation | 02 §6.1 | 🔴 | none — index walks and custom sorts | — |
| 4.2.1 | Stack matching and evaluation | 03 §1.1 | 🟡 | for each token: push, or pop and combine | what you push and what triggers a pop |
| 4.2.2 | Monotonic stack | 03 §1.2 | 🟢 | `while stack and top < x: pop and resolve`; push | comparison direction and what a pop records |
| 4.2.3 | Classic binary search | 03 §2.1 | 🟢 | `first_true(lo, hi, is_true)` | only `is_true` (and the range) |
| 4.2.4 | Rotated arrays | 03 §2.2 | 🟡 | compare `nums[mid]` with `nums[hi]` to find the sorted half | what you do once you know which half is sorted |
| 4.2.5 | Binary search on the answer | 03 §2.3 | 🟢 | `min_feasible(lo, hi, feasible)` | `feasible(x)` and the range |
| 4.2.6 | Linked list reversal and merging | 03 §3.1 | 🟡 | 3-line reverse · dummy + tail merge | whole list vs one segment; two lists vs k |
| 4.2.7 | Slow and fast pointers | 03 §3.2 | 🟢 | slow moves 1, fast moves 2 (or gets a head start) | the stop condition and what you return |
| 4.2.8 | Linked list rewiring | 03 §3.3 | 🔴 | none — old→new map | — |
| 4.2.9 | Linked list design (LRU) | 03 §3.4 | 🔴 | none — hash map + doubly linked list | — |
| 4.3.1 | Tree bottom-up recursion | 04 §1.1 | 🟢 | `dfs(node)`: base → recurse both → update best → return | empty value, the `best` update, the return |
| 4.3.2 | Tree top-down paths | 04 §1.2 | 🟢 | `dfs(node, state)`: update state, check, pass down | what the state is and when you record |
| 4.3.3 | Tree construction and serialisation | 04 §1.3 | 🔴 | none — five separate ideas | — |
| 4.3.4 | Tree level-order BFS | 04 §2.1 | 🟢 | `for _ in range(len(queue))` = one level | what you do with each level |
| 4.3.5 | BST in-order and bounds | 04 §3.1 | 🟡 | in-order walk (sorted order) · walk down by comparing | need sorted order → in-order; need one key → walk down |
| 4.3.6 | Trie (prefix tree) | 04 §4.1 | 🟢 | nested dicts + an end marker | how you walk it (exact, wildcard, alongside a grid) |
| 4.3.7 | Heap top-k and k-th element | 04 §5.1 | 🟢 | size-k min-heap · quickselect | the key (negate to flip), or a frontier of pairs |
| 4.3.8 | Heap scheduling and simulation | 04 §5.2 | 🟡 | sort by one key + heap on the other · two heaps · greedy max | which of 3 shapes |
| 4.4.1 | Grid connected regions (DFS) | 05 §1.1 | 🟢 | for each start cell: flood its region and mark it | which cells are starts, what the flood returns |
| 4.4.2 | Grid shortest paths (BFS) | 05 §1.2 | 🟢 | multi-source BFS, one ring per step | the sources, the passable test, when to stop |
| 4.4.3 | Adjacency-list traversal | 05 §2.1 | 🟡 | build graph → visited set → DFS/BFS | graph shape (directed / weighted) and what a visit does |
| 4.4.4 | BFS over state spaces | 05 §2.2 | 🟢 | BFS where `neighbours(state)` generates the edges | only `neighbours(state)` |
| 4.4.5 | Topological sort | 05 §3.1 | 🟢 | Kahn: queue of in-degree 0, peel, decrement | how edges are built; what you return |
| 4.4.6 | Union-Find | 05 §4.1 | 🟢 | `UnionFind` class (find with path halving, union by size) | what you union and what you read back |
| 4.4.7 | Dijkstra and variants | 05 §5.1 | 🟡 | Dijkstra · Bellman–Ford (k rounds) · Prim / Kruskal | how a path's cost combines; stop limit → Bellman–Ford |
| 4.4.8 | Subsets, combinations, permutations | 05 §6.1 | 🟢 | choose → explore → un-choose, from `start` | when to record, `i` vs `i + 1`, skip-dup, `used[]` |
| 4.4.9 | Backtracking with constraints | 05 §6.2 | 🟡 | backtrack + `is_valid` check before each choice | the choice list and the validity test |
| 4.5.1 | Fibonacci-style recurrences | 06 §1.1 | 🟢 | roll two variables: `prev2, prev1 = prev1, new` | the one line that computes `new` |
| 4.5.2 | Kadane and subarray DP | 06 §1.2 | 🟢 | `cur = max(x, cur + x)`; best = max(best, cur) | what you track alongside `cur` |
| 4.5.3 | Word and partition DP | 06 §1.3 | 🟡 | `dp[i]` = combine over every last piece | what a 'last piece' is and how you combine (any / min / max / sum) |
| 4.5.4 | Longest increasing subsequence | 06 §1.4 | 🟢 | `tails` + `bisect_left` (or O(n²) `dp[i]`) | `bisect_left` (strict) vs `bisect_right` (non-decreasing) |
| 4.5.5 | Knapsack: subset sums and coins | 06 §2.1 | 🟢 | 1D `dp` over capacity | loop direction, loop order, combine (min / + / or) |
| 4.5.6 | State machine DP (stocks) | 06 §3.1 | 🟢 | one variable per state, all updated each day | add a state or a fee in the transition |
| 4.5.7 | Grid DP | 06 §4.1 | 🟢 | `dp[r][c]` from up and left (one rolling row) | the combine line and the first row/column |
| 4.5.8 | Palindrome DP | 06 §5.1 | 🟡 | expand around centre · interval DP | substring → expand; subsequence → `dp[i][j]` |
| 4.5.9 | Two-string alignment DP | 06 §5.2 | 🟢 | `dp[i][j]` over prefixes `a[:i]`, `b[:j]` | the match and mismatch lines (+ base row/column) |
| 4.5.10 | Interval and game DP | 06 §6.1 | 🟢 | `dp(i, j)` = best score lead for the player to move | nothing between 486 and 877 — only the final check |
| 4.6.1 | Greedy reachability and scans | 07 §1.1 | 🔴 | none — a handful of one-pass ideas | — |
| 4.6.2 | Intervals: merging and scheduling | 07 §2.1 | 🟡 | sort by start (merge) · sort by end (pick) · heap or sweep (overlap) | the sort key and what an overlap does |
| 4.6.3 | Number manipulation | 07 §3.1 | 🔴 | none — digit loop, carry walk, fast power | — |
| 4.6.4 | Geometry and randomness | 07 §3.2 | 🔴 | none — three separate ideas | — |
| 4.6.5 | Bit manipulation | 07 §4.1 | 🔴 | none — a trick sheet | — |
| 4.6.6 | Data-structure design | 07 §5.1 | 🔴 | none — pick one structure per operation | — |

## 4.1 Arrays and Strings

### 4.1.1 Hash map and set lookups · 🟡

**Practice:** 02 §1.1 · **Template:** seen-before · count-and-compare · group-by-key

Three small shapes. First decide which one the question is, then change the key.

| Shape | Questions | What you change |
|---|---|---|
| Seen-before (one pass) | 1, 217, 219, 128 | the partner you look up (`target - x`, `x`, `x - 1`) and what you store (index, last index) |
| Count and compare | 242, 383, 409, 347, 169, 229, 448 | what you count and the final check (`==`, `<=`, odd counts, top k) |
| Group by key | 49, 36 | the key (`sorted(word)`, `(row, digit)`, `(box, digit)`) |

Off-template: 238 (prefix × suffix pass), 271 (length-prefix encoding), 12 (walk a value table), 274 (sort, then scan).

In [5]:
def seen_before(nums, target):
    """Seen-before: look up the partner, then remember x."""
    seen = {}
    for i, x in enumerate(nums):
        # CHANGE: the partner you need (target - x, x itself, x - 1 ...)
        need = target - x
        if need in seen:
            return [seen[need], i]
        # CHANGE: what you remember about x (index, last index, count)
        seen[x] = i
    return []


def count_and_compare(a, b):
    """Count-and-compare: build counts, then one check."""
    # CHANGE: what you count
    need, have = Counter(a), Counter(b)
    # CHANGE: the check (== anagram, <= ransom note, most_common(k) top-k)
    return need == have


def group_by_key(items):
    """Group-by-key: equivalent items collide on the same key."""
    groups = defaultdict(list)
    for item in items:
        # CHANGE: the key that makes 'equivalent' items equal
        groups["".join(sorted(item))].append(item)
    return list(groups.values())


print(seen_before([2, 7, 11, 15], 9), count_and_compare("anagram", "nagaram"))
print(group_by_key(["eat", "tea", "tan", "ate", "nat", "bat"]))

[0, 1] True
[['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]


### 4.1.2 In-place array manipulation · 🟡

**Practice:** 02 §1.2 · **Template:** write pointer · 3-way partition · reverse pieces

Pick the shape from what "in place" means here.

| Shape | Questions | What you change |
|---|---|---|
| Write pointer (keep / skip) | 26, 80, 283 | the keep rule (`!= nums[w-1]`, `!= nums[w-2]`, `!= 0`) |
| Fill from the back | 88 | — (write the bigger tail first) |
| Three-way partition | 75, 912 (quicksort) | the pivot |
| Reverse pieces | 189, 31 | which ranges you reverse |

In [6]:
def compact(nums):
    """Write pointer: keep wanted values at the front, return new length."""
    write = 0
    for x in nums:
        # CHANGE: keep rule (26: != nums[write-1]; 80: != nums[write-2]; 283: != 0)
        if write < 1 or nums[write - 1] != x:
            nums[write] = x
            write += 1
    return write


def three_way_partition(nums, pivot):
    """Dutch flag: < pivot | == pivot | > pivot in one pass."""
    low, mid, high = 0, 0, len(nums) - 1
    while mid <= high:
        if nums[mid] < pivot:
            nums[low], nums[mid] = nums[mid], nums[low]
            low += 1
            mid += 1
        elif nums[mid] > pivot:
            # mid stays: the value swapped in is not checked yet
            nums[mid], nums[high] = nums[high], nums[mid]
            high -= 1
        else:
            mid += 1
    return nums


def reverse_range(nums, lo, hi):
    """Reverse nums[lo..hi] in place."""
    while lo < hi:
        nums[lo], nums[hi] = nums[hi], nums[lo]
        lo += 1
        hi -= 1


def rotate_right(nums, k):
    """Rotate = reverse all, then reverse each piece."""
    k %= len(nums)
    reverse_range(nums, 0, len(nums) - 1)
    reverse_range(nums, 0, k - 1)
    reverse_range(nums, k, len(nums) - 1)
    return nums


nums = [1, 1, 2, 3, 3]
print(compact(nums), nums[:3], three_way_partition([2, 0, 2, 1, 1, 0], 1))
print(rotate_right([1, 2, 3, 4, 5, 6, 7], 3))

3 [1, 2, 3] [0, 0, 1, 1, 2, 2]
[5, 6, 7, 1, 2, 3, 4]


### 4.1.3 Opposite-end pointers · 🟢

**Practice:** 02 §2.1 · **Template:** `while left < right`: look at both ends, move one

One loop. Every question changes the same three lines.

| Question | What you change |
|---|---|
| 125 Valid Palindrome | skip non-alphanumeric; mismatch → `False`; move both |
| 167 Two Sum II | sum too small → `left += 1`, else `right -= 1` |
| 15 3Sum · 18 4Sum | wrap in a loop over `i` (and `j`); skip equal neighbours |
| 11 Container With Most Water | area = width × shorter wall; move the shorter wall |
| 42 Trapping Rain Water | keep `left_max` / `right_max`; move the lower side, add water |
| 680 Valid Palindrome II | on first mismatch, try skipping `left` or `right` once |
| 881 Boats to Save People | sort; heaviest boards, lightest joins if it fits |

In [7]:
def two_sum_sorted(nums, target):
    """Template: close in from both ends."""
    left, right = 0, len(nums) - 1
    while left < right:
        # CHANGE: what you compute from the two ends
        total = nums[left] + nums[right]
        # CHANGE: when you are done / what you record
        if total == target:
            return [left, right]
        # CHANGE: which side to move
        if total < target:
            left += 1
        else:
            right -= 1
    return []


def max_area(heights):
    """Same loop for 11: three lines changed."""
    left, right, best = 0, len(heights) - 1, 0
    while left < right:
        best = max(best, (right - left) * min(heights[left], heights[right]))
        if heights[left] < heights[right]:
            left += 1
        else:
            right -= 1
    return best


print(two_sum_sorted([2, 7, 11, 15], 9), max_area([1, 8, 6, 2, 5, 4, 8, 3, 7]))

[0, 1] 49


### 4.1.4 Same-direction pointers · 🟢

**Practice:** 02 §2.2 · **Template:** `i` walks one sequence, `j` follows / reads ahead

| Question | What you change |
|---|---|
| 392 Is Subsequence | `i` on `s` moves only on a match; `j` on `t` always moves |
| 151 Reverse Words | `i` skips spaces, `j` reads one word, collect, reverse at the end |

In [8]:
def is_subsequence(s, t):
    """i follows j only when characters match."""
    i = 0
    for ch in t:
        # CHANGE: when the slow pointer moves
        if i < len(s) and s[i] == ch:
            i += 1
    return i == len(s)


def read_words(s):
    """i skips spaces, j reads one word."""
    words, i = [], 0
    while i < len(s):
        if s[i] == " ":
            i += 1
            continue
        j = i
        while j < len(s) and s[j] != " ":
            j += 1
        words.append(s[i:j])
        i = j
    return words


print(
    is_subsequence("abc", "ahbgdc"),
    " ".join(reversed(read_words("  the sky  is blue "))),
)

True blue is sky the


### 4.1.5 Variable-size windows · 🟢

**Practice:** 02 §3.1 · **Template:** extend right → shrink while broken → record

Two versions: **longest** (shrink while broken, record after) and **shortest** (shrink while good, record inside).

| Question | Version | What you change |
|---|---|---|
| 3 Longest Substring w/o Repeat | longest | broken: `count[ch] > 1` |
| 424 Longest Repeating Replacement | longest | broken: `size - max_freq > k` |
| 1004 Max Consecutive Ones III | longest | broken: `zeros > k` |
| 1493 Longest 1's After Deleting One | longest | 1004 with `k = 1`, answer − 1 |
| 904 Fruit Into Baskets | longest | broken: `len(count) > 2` |
| 209 Minimum Size Subarray Sum | shortest | good: `total >= target` |
| 76 Minimum Window Substring | shortest | good: all needed chars covered (`formed == len(need)`) |
| 121 Best Time to Buy/Sell | — | window collapses to "min price so far" |

In [9]:
def longest_window(s, k):
    """Longest window with at most k distinct chars."""
    count = Counter()
    left = best = 0
    for right, ch in enumerate(s):
        # CHANGE: add s[right] to the window state
        count[ch] += 1
        # CHANGE: the 'window broken' condition
        while len(count) > k:
            count[s[left]] -= 1
            if count[s[left]] == 0:
                del count[s[left]]
            left += 1
        best = max(best, right - left + 1)
    return best


def shortest_window(nums, target):
    """Shortest window with sum >= target: shrink while still good."""
    total, left, best = 0, 0, float("inf")
    for right, x in enumerate(nums):
        total += x
        # CHANGE: the 'window good' condition; record before shrinking
        while total >= target:
            best = min(best, right - left + 1)
            total -= nums[left]
            left += 1
    return 0 if best == float("inf") else best


print(longest_window("eceba", 2), shortest_window([2, 3, 1, 2, 4, 3], 7))

3 2


### 4.1.6 Fixed-size windows · 🟢

**Practice:** 02 §3.2 · **Template:** add `right`, drop `right - k`, check once full

| Question | What you change |
|---|---|
| 567 Permutation in String · 438 Find All Anagrams | state = Counter of the last k chars; check `window == need` |
| 1423 Max Points from Cards | keep the middle `n - k` cards: minimum window sum of size `n - k` |
| 239 Sliding Window Maximum | state = monotonic deque of indices (front = max) |

In [10]:
def anagram_starts(s, p):
    """Start indices of size-k windows matching p's letter counts."""
    k, need = len(p), Counter(p)
    window, hits = Counter(), []
    for right, ch in enumerate(s):
        # add the new right element
        window[ch] += 1
        # drop the element that just left (index right - k)
        if right >= k:
            out = s[right - k]
            window[out] -= 1
            if window[out] == 0:
                del window[out]
        # CHANGE: what you check once the window is full
        if right >= k - 1 and window == need:
            hits.append(right - k + 1)
    return hits


def window_max(nums, k):
    """239: same frame, state is a decreasing deque of indices."""
    dq, out = deque(), []
    for i, x in enumerate(nums):
        while dq and nums[dq[-1]] <= x:
            dq.pop()
        dq.append(i)
        if dq[0] <= i - k:
            dq.popleft()
        if i >= k - 1:
            out.append(nums[dq[0]])
    return out


print(anagram_starts("cbaebabacd", "abc"), window_max([1, 3, -1, -3, 5, 3, 6, 7], 3))

[0, 6] [3, 3, 5, 5, 6, 7]


### 4.1.7 Prefix sum with hash map · 🟢

**Practice:** 02 §4.1 · **Template:** running prefix + map of earlier prefixes

Store **counts** when you count subarrays, **first index** when you want the longest.

| Question | What you change |
|---|---|
| 560 Subarray Sum Equals K | count; look up `prefix - k` |
| 525 Contiguous Array | map 0 → −1, 1 → +1; first index; look up the same prefix |
| 523 Continuous Subarray Sum | key = `prefix % k`; first index; length ≥ 2 |

In [11]:
def count_subarrays(nums, k):
    """Count subarrays summing to k."""
    seen = Counter({0: 1})
    prefix = total = 0
    for x in nums:
        # CHANGE: how x moves the prefix (x, +1/-1, then % k)
        prefix += x
        # CHANGE: the earlier prefix that pairs with this one
        total += seen[prefix - k]
        seen[prefix] += 1
    return total


def longest_zero_sum(nums):
    """Longest subarray with sum 0: remember the FIRST index of each prefix."""
    first = {0: -1}
    prefix = best = 0
    for i, x in enumerate(nums):
        prefix += x
        if prefix in first:
            best = max(best, i - first[prefix])
        else:
            first[prefix] = i
    return best


# 525: turn 0 into -1, then 'equal 0s and 1s' == 'sum 0'
print(
    count_subarrays([1, 1, 1], 2), longest_zero_sum([1 if b else -1 for b in [0, 1, 0]])
)

2 2


### 4.1.8 Range queries and difference arrays · 🟡

**Practice:** 02 §4.2 · **Template:** prefix (reads) · difference (updates) · Fenwick (both)

Two questions decide the template: **are you reading ranges, adding to ranges, or doing both mixed?** and **1D or 2D?**

| Question | Choice | What you change |
|---|---|---|
| 304 Range Sum Query 2D | read · 2D | prefix grid + inclusion–exclusion |
| 1109 Flight Bookings · 1094 Car Pooling | update · 1D | `+v` at start, `-v` after end, then running sum (1094: check ≤ capacity) |
| 2536 Increment Submatrices | update · 2D | four corners, then 2D running sum |
| 528 Random Pick with Weight | read · 1D | prefix of weights + `bisect_left` on a random ticket |
| 307 Range Sum Query – Mutable | read + update mixed · 1D | Fenwick tree: hop by `i & -i`; `sum(l, r) = prefix(r + 1) - prefix(l)` |

In [12]:
def prefix_2d(grid):
    """pre[r][c] = sum of grid[:r][:c] (one extra row and column of 0)."""
    rows, cols = len(grid), len(grid[0])
    pre = [[0] * (cols + 1) for _ in range(rows + 1)]
    for r in range(rows):
        for c in range(cols):
            pre[r + 1][c + 1] = grid[r][c] + pre[r][c + 1] + pre[r + 1][c] - pre[r][c]
    return pre


def region_sum(pre, r1, c1, r2, c2):
    """Inclusion-exclusion on the prefix grid."""
    return pre[r2 + 1][c2 + 1] - pre[r1][c2 + 1] - pre[r2 + 1][c1] + pre[r1][c1]


def range_adds(n, updates):
    """1D difference array: (start, end, value) adds value to [start, end]."""
    diff = [0] * (n + 1)
    for start, end, value in updates:
        diff[start] += value
        diff[end + 1] -= value
    out, running = [], 0
    for i in range(n):
        running += diff[i]
        out.append(running)
    return out


def box_adds(n, boxes):
    """2D difference array: +1 on each box, then a 2D prefix sum."""
    diff = [[0] * (n + 1) for _ in range(n + 1)]
    for r1, c1, r2, c2 in boxes:
        diff[r1][c1] += 1
        diff[r1][c2 + 1] -= 1
        diff[r2 + 1][c1] -= 1
        diff[r2 + 1][c2 + 1] += 1
    # running sums: rows first, then columns
    for r in range(n):
        for c in range(1, n):
            diff[r][c] += diff[r][c - 1]
    for r in range(1, n):
        for c in range(n):
            diff[r][c] += diff[r - 1][c]
    return [row[:n] for row in diff[:n]]


class Fenwick:
    """Point update + prefix sum, both O(log n) (1-indexed inside)."""

    def __init__(self, n):
        self.tree = [0] * (n + 1)

    def add(self, i, delta):
        # climb: every block that covers position i
        i += 1
        while i < len(self.tree):
            self.tree[i] += delta
            i += i & -i

    def prefix(self, i):
        """Sum of the first i values."""
        total = 0
        # hop down: drop the lowest set bit each step
        while i > 0:
            total += self.tree[i]
            i -= i & -i
        return total


pre = prefix_2d([[3, 0, 1], [5, 6, 3], [1, 2, 0]])
print(region_sum(pre, 1, 1, 2, 2), range_adds(5, [(0, 1, 10), (1, 2, 20), (1, 4, 25)]))
print(box_adds(3, [(1, 1, 2, 2), (0, 0, 1, 1)]))
# 528: weights [1, 3] -> prefix [1, 4]; ticket 2 lands in bucket 1
print(bisect_left([1, 4], 2))
# 307: [1, 3, 5], then set index 1 to 2
fen = Fenwick(3)
for i, x in enumerate([1, 3, 5]):
    fen.add(i, x)
fen.add(1, 2 - 3)
print(fen.prefix(3) - fen.prefix(0))

11 [10, 55, 45, 25, 25]
[[1, 1, 0], [1, 2, 1], [0, 1, 1]]
1
8


### 4.1.9 Matrix traversal and transforms · 🔴

**Practice:** 02 §5.1 · **Template:** none — four tricks

No shared template. Each question is one trick:

| Question | Trick |
|---|---|
| 54 Spiral Matrix | four boundaries (`top`, `bottom`, `left`, `right`) that shrink after each side |
| 48 Rotate Image | rotate clockwise = transpose, then reverse each row |
| 73 Set Matrix Zeroes | use row 0 / column 0 as the flags (plus one flag for row 0 itself) |
| 289 Game of Life | store old and new state in the same cell (bit 0 = now, bit 1 = next) |

In [13]:
def spiral(matrix):
    """Peel the outer ring, shrink the boundaries, repeat."""
    out = []
    top, bottom, left, right = 0, len(matrix) - 1, 0, len(matrix[0]) - 1
    while top <= bottom and left <= right:
        out += [matrix[top][c] for c in range(left, right + 1)]
        out += [matrix[r][right] for r in range(top + 1, bottom + 1)]
        if top < bottom:
            out += [matrix[bottom][c] for c in range(right - 1, left - 1, -1)]
        if left < right:
            out += [matrix[r][left] for r in range(bottom - 1, top, -1)]
        top, bottom, left, right = top + 1, bottom - 1, left + 1, right - 1
    return out


def rotate(matrix):
    """Clockwise = transpose + reverse each row."""
    n = len(matrix)
    for r in range(n):
        for c in range(r + 1, n):
            matrix[r][c], matrix[c][r] = matrix[c][r], matrix[r][c]
    for row in matrix:
        row.reverse()
    return matrix


def live_neighbours(board, r, c):
    """8-direction neighbour count; '& 1' reads the OLD state."""
    total = 0
    for dr in (-1, 0, 1):
        for dc in (-1, 0, 1):
            nr, nc = r + dr, c + dc
            if (dr or dc) and 0 <= nr < len(board) and 0 <= nc < len(board[0]):
                total += board[nr][nc] & 1
    return total


print(spiral([[1, 2, 3], [4, 5, 6], [7, 8, 9]]), rotate([[1, 2], [3, 4]]))
print(live_neighbours([[0, 1, 0], [0, 0, 1], [1, 1, 1]], 1, 1))

[1, 2, 3, 6, 9, 8, 7, 4, 5] [[3, 1], [4, 2]]
5


### 4.1.10 String parsing and simulation · 🔴

**Practice:** 02 §6.1 · **Template:** none — index walks and custom sorts

No shared template. The reusable ideas:

| Idea | Questions |
|---|---|
| Walk an index `i` through the string, one phase at a time (spaces → sign → digits) | 8 atoi |
| Run-length groups: `j` runs while `s[j] == s[i]` | 443 String Compression |
| Custom sort with `cmp_to_key` | 179 Largest Number (`a + b` vs `b + a`) |
| Bounce a row index between 0 and `rows - 1` | 6 Zigzag Conversion |
| Split, pad the shorter side with 0, compare piece by piece | 165 Compare Version Numbers |

In [14]:
def run_groups(s):
    """Run-length groups: (char, run length)."""
    out, i = [], 0
    while i < len(s):
        j = i
        while j < len(s) and s[j] == s[i]:
            j += 1
        out.append((s[i], j - i))
        i = j
    return out


def largest_number(nums):
    """Put a before b if a+b reads bigger than b+a."""
    words = [str(x) for x in nums]
    words.sort(key=cmp_to_key(lambda a, b: -1 if a + b > b + a else 1))
    return "".join(words).lstrip("0") or "0"


def my_atoi(s):
    """Index walk: spaces, sign, digits, clamp."""
    i, n, sign, value = 0, len(s), 1, 0
    while i < n and s[i] == " ":
        i += 1
    if i < n and s[i] in "+-":
        sign = -1 if s[i] == "-" else 1
        i += 1
    while i < n and s[i].isdigit():
        value = value * 10 + int(s[i])
        i += 1
    return max(-(2**31), min(2**31 - 1, sign * value))


print(run_groups("aabccc"), largest_number([3, 30, 34, 5, 9]), my_atoi("  -42abc"))

[('a', 2), ('b', 1), ('c', 3)] 9534330 -42


## 4.2 Stack, Binary Search, Linked List

### 4.2.1 Stack matching and evaluation · 🟡

**Practice:** 03 §1.1 · **Template:** for each token: push, or pop and combine

Same loop; the condition table is what you design.

| Question | Push | Pop when |
|---|---|---|
| 20 Valid Parentheses | openers | a closer arrives (must match the top) |
| 150 Evaluate RPN | numbers | an operator arrives (pop two, push result) |
| 71 Simplify Path | folder names | `..` |
| 2390 Removing Stars | characters | `*` |
| 735 Asteroid Collision | asteroids | a left-mover hits a smaller right-mover |
| 1249 Min Remove to Make Valid | indices of `(` | `)` matches; leftovers are removed |
| 227 Basic Calculator II | signed numbers | `*` or `/` combines with the top |
| 394 Decode String · 224 Basic Calculator | the current context on `[` / `(` | `]` / `)`: pop context and combine |
| 155 Min Stack · 232 Queue via Stacks | design: a second stack (running min / output stack) | — |

In [15]:
def is_valid(s):
    """Push openers, pop on closers."""
    pairs = {")": "(", "]": "[", "}": "{"}
    stack = []
    for ch in s:
        # CHANGE: the pop trigger and the check
        if ch in pairs:
            if not stack or stack.pop() != pairs[ch]:
                return False
        else:
            stack.append(ch)
    return not stack


def decode_string(s):
    """Nested: push the current context on '[', pop and combine on ']'."""
    stack, current, number = [], "", 0
    for ch in s:
        if ch.isdigit():
            number = number * 10 + int(ch)
        elif ch == "[":
            stack.append((current, number))
            current, number = "", 0
        elif ch == "]":
            before, times = stack.pop()
            current = before + current * times
        else:
            current += ch
    return current


print(is_valid("([]{})"), is_valid("(]"), decode_string("3[a2[c]]"))

True False accaccacc


### 4.2.2 Monotonic stack · 🟢

**Practice:** 03 §1.2 · **Template:** `while stack and top < x: pop and resolve`; push

| Question | What you change |
|---|---|
| 739 Daily Temperatures | on pop: `answer[j] = i - j` |
| 901 Online Stock Span | stack of `(price, span)`; popping adds spans |
| 853 Car Fleet | sort by position desc; stack of arrival times; push only slower cars |
| 84 Largest Rectangle | increasing stack; on pop: width = `i - stack[-1] - 1`; add a 0 at the end |
| 402 Remove K Digits | pop bigger digits while `k > 0` |

In [16]:
def wait_days(temps):
    """Template: each index waits on the stack until something beats it."""
    answer, stack = [0] * len(temps), []
    for i, x in enumerate(temps):
        # CHANGE: comparison (< next greater, > next smaller)
        while stack and temps[stack[-1]] < x:
            j = stack.pop()
            # CHANGE: what a pop records
            answer[j] = i - j
        stack.append(i)
    return answer


def largest_rectangle(heights):
    """84: increasing stack; a pop means 'this bar can extend no further'."""
    stack, best = [], 0
    for i, h in enumerate(heights + [0]):
        while stack and heights[stack[-1]] > h:
            height = heights[stack.pop()]
            width = i if not stack else i - stack[-1] - 1
            best = max(best, height * width)
        stack.append(i)
    return best


print(
    wait_days([73, 74, 75, 71, 69, 72, 76, 73]), largest_rectangle([2, 1, 5, 6, 2, 3])
)

[1, 1, 4, 2, 1, 1, 0, 0] 10


### 4.2.3 Classic binary search · 🟢

**Practice:** 03 §2.1 · **Template:** `first_true(lo, hi, is_true)`

Turn every question into "first index where something becomes true".

| Question | `is_true(i)` / what you change |
|---|---|
| 704 Binary Search · 35 Search Insert | `nums[i] >= target` (704: then check equality) |
| 278 First Bad Version | `is_bad(i)` |
| 34 First and Last Position | `>= target` and `>= target + 1`, minus 1 |
| 74 Search a 2D Matrix | flat index `i` → `(i // cols, i % cols)` |
| 69 Sqrt(x) | first `i` with `i * i > x`, answer − 1 |
| 162 Find Peak Element | `nums[i] > nums[i + 1]` |
| 540 Single Element | pair at `i` is broken: `nums[i] != nums[i ^ 1]` |
| 658 Find K Closest | first `left` with `x - a[left] <= a[left + k] - x` |
| 2300 Spells and Potions | sort potions; first with `potion * spell >= success` |
| 240 Search a 2D Matrix II | not binary search: staircase from the top-right corner |

In [17]:
def first_true(lo, hi, is_true):
    """Smallest i in [lo, hi) with is_true(i); hi if none. is_true must go F..FT..T."""
    while lo < hi:
        mid = (lo + hi) // 2
        if is_true(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo


nums = [1, 2, 2, 2, 5]
# 35: insert position
print(first_true(0, len(nums), lambda i: nums[i] >= 3))
# 34: first and last 2
print(
    first_true(0, len(nums), lambda i: nums[i] >= 2),
    first_true(0, len(nums), lambda i: nums[i] >= 3) - 1,
)
# 69: sqrt(8)
print(first_true(0, 9, lambda i: i * i > 8) - 1)
# 162: a peak
peaks = [1, 2, 1, 3, 5, 6, 4]
print(first_true(0, len(peaks) - 1, lambda i: peaks[i] > peaks[i + 1]))

4
1 3
2
5


### 4.2.4 Rotated arrays · 🟡

**Practice:** 03 §2.2 · **Template:** compare `nums[mid]` with `nums[hi]` to find the sorted half

The extra condition: **which half is sorted?** `nums[mid] > nums[hi]` → the left half is sorted and the drop is on the right.

| Question | Condition you add |
|---|---|
| 153 Find Minimum | `nums[mid] > nums[hi]` → go right, else `hi = mid` |
| 33 Search in Rotated | find the sorted half; is target inside its range? go there, else the other half |
| 81 Search in Rotated II | duplicates: if `nums[mid] == nums[hi]`, just `hi -= 1` |

In [18]:
def find_min_rotated(nums):
    """Shrink toward the drop point."""
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] > nums[hi]:
            lo = mid + 1
        else:
            hi = mid
    return nums[lo]


def search_rotated(nums, target):
    """One half is always sorted; check if target lives there."""
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return mid
        # left half sorted
        if nums[lo] <= nums[mid]:
            if nums[lo] <= target < nums[mid]:
                hi = mid - 1
            else:
                lo = mid + 1
        # right half sorted
        else:
            if nums[mid] < target <= nums[hi]:
                lo = mid + 1
            else:
                hi = mid - 1
    return -1


print(find_min_rotated([4, 5, 6, 7, 0, 1, 2]), search_rotated([4, 5, 6, 7, 0, 1, 2], 0))

0 4


### 4.2.5 Binary search on the answer · 🟢

**Practice:** 03 §2.3 · **Template:** `min_feasible(lo, hi, feasible)`

Write `feasible(x)` (is answer `x` good enough?), pick the range, done.

| Question | Range | `feasible(x)` |
|---|---|---|
| 875 Koko Eating Bananas | `[1, max(piles)]` | `sum(ceil(p / x)) <= h` |
| 1011 Ship Packages in D Days | `[max(w), sum(w)]` | days needed with capacity `x` `<= days` |
| 378 Kth Smallest in Sorted Matrix | `[min, max]` | count of values `<= x` is `>= k` |
| 981 Time Based Key-Value Store | — | `bisect_right` on the timestamps of a key |
| 4 Median of Two Sorted Arrays | off-template: binary search the cut in the shorter array | |

In [19]:
def min_feasible(lo, hi, feasible):
    """Smallest x in [lo, hi] with feasible(x) True."""
    while lo < hi:
        mid = (lo + hi) // 2
        if feasible(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo


def koko(piles, h):
    """875."""
    # CHANGE: the feasibility test and the range
    return min_feasible(1, max(piles), lambda k: sum(-(-p // k) for p in piles) <= h)


def ship_capacity(weights, days):
    """1011."""

    def fits(cap):
        used, load = 1, 0
        for w in weights:
            if load + w > cap:
                used, load = used + 1, 0
            load += w
        return used <= days

    return min_feasible(max(weights), sum(weights), fits)


print(koko([3, 6, 7, 11], 8), ship_capacity([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 5))

4 15


### 4.2.6 Linked list reversal and merging · 🟡

**Practice:** 03 §3.1 · **Template:** 3-line reverse · dummy + tail merge

Two building blocks. The condition is **how much** you reverse or **how many** lists you merge.

| Question | Build from |
|---|---|
| 206 Reverse Linked List | reverse |
| 92 Reverse Linked List II | walk to the node before `left`, reverse `right - left + 1` nodes, reconnect |
| 25 Reverse Nodes in k-Group | 92 repeated for every full block of k |
| 21 Merge Two Sorted Lists | dummy + tail, take the smaller head |
| 23 Merge k Sorted Lists | heap of `(value, i, node)` heads |
| 148 Sort List | split at the middle (slow/fast), sort both halves, merge two |
| 2 Add Two Numbers | dummy + tail, carry instead of comparison |

In [20]:
class ListNode:
    """Singly linked list node."""

    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next


def from_list(values):
    """Linked list from a Python list."""
    dummy = tail = ListNode()
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next


def to_list(head):
    """Python list from a linked list."""
    out = []
    while head:
        out.append(head.val)
        head = head.next
    return out


def reverse(head):
    """Point every arrow backwards."""
    prev = None
    while head:
        head.next, prev, head = prev, head, head.next
    return prev


def merge_two(a, b):
    """Dummy + tail: always attach the smaller head."""
    dummy = tail = ListNode()
    while a and b:
        if a.val <= b.val:
            tail.next, a = a, a.next
        else:
            tail.next, b = b, b.next
        tail = tail.next
    tail.next = a or b
    return dummy.next


def merge_k(lists):
    """Heap of current heads; i breaks ties so nodes are never compared."""
    heap = [(node.val, i, node) for i, node in enumerate(lists) if node]
    heapq.heapify(heap)
    dummy = tail = ListNode()
    while heap:
        _, i, node = heapq.heappop(heap)
        tail.next = tail = node
        if node.next:
            heapq.heappush(heap, (node.next.val, i, node.next))
    return dummy.next


print(
    to_list(reverse(from_list([1, 2, 3]))),
    to_list(merge_two(from_list([1, 4]), from_list([2, 3]))),
)
print(to_list(merge_k([from_list([1, 4, 5]), from_list([1, 3, 4]), from_list([2, 6])])))

[3, 2, 1] [1, 2, 3, 4]
[1, 1, 2, 3, 4, 4, 5, 6]


### 4.2.7 Slow and fast pointers · 🟢

**Practice:** 03 §3.2 · **Template:** slow moves 1, fast moves 2 (or gets a head start)

| Question | What you change |
|---|---|
| 876 Middle of the List | return `slow` |
| 141 Linked List Cycle | `slow is fast` → cycle |
| 142 Linked List Cycle II | after they meet, restart one at head; step both by 1 until equal |
| 287 Find the Duplicate Number | 142 with `next = nums[i]` |
| 19 Remove Nth From End | fast gets an `n`-step head start; stop at `fast.next` |
| 143 Reorder List | middle → reverse second half → weave |

In [21]:
def middle(head):
    """Template: fast reaches the end when slow is halfway."""
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        # CHANGE: extra check inside the loop (141: if slow is fast: return True)
    # CHANGE: what you return
    return slow


def cycle_start(head):
    """142: meet, then walk from head and meeting point at the same speed."""
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        if slow is fast:
            slow = head
            while slow is not fast:
                slow, fast = slow.next, fast.next
            return slow
    return None


def remove_nth_from_end(head, n):
    """19: fast gets n steps ahead, then both move."""
    dummy = ListNode(0, head)
    slow = fast = dummy
    for _ in range(n):
        fast = fast.next
    while fast.next:
        slow, fast = slow.next, fast.next
    slow.next = slow.next.next
    return dummy.next


loop = from_list([3, 2, 0, -4])
loop.next.next.next.next = loop.next
print(middle(from_list([1, 2, 3, 4, 5])).val, cycle_start(loop).val)
print(to_list(remove_nth_from_end(from_list([1, 2, 3, 4, 5]), 2)))

3 2
[1, 2, 3, 5]


### 4.2.8 Linked list rewiring · 🔴

**Practice:** 03 §3.3 · **Template:** none — old→new map

One question, one idea: **map every old node to its copy**, then wire `next` and `random` through the map. (The O(1)-space follow-up interleaves copies: `A → A' → B → B'`.)

In [22]:
class RandomNode:
    """Node with an extra random pointer."""

    def __init__(self, val, next=None, random=None):
        self.val = val
        self.next = next
        self.random = random


def copy_random_list(head):
    """Pass 1: make copies. Pass 2: wire pointers through the map."""
    copy = {None: None}
    node = head
    while node:
        copy[node] = RandomNode(node.val)
        node = node.next
    node = head
    while node:
        copy[node].next = copy[node.next]
        copy[node].random = copy[node.random]
        node = node.next
    return copy[head]


a, b = RandomNode(7), RandomNode(13)
a.next, b.random = b, a
c = copy_random_list(a)
print(c.val, c.next.val, c.next.random.val, c is not a)

7 13 7 True


### 4.2.9 Linked list design (LRU) · 🔴

**Practice:** 03 §3.4 · **Template:** none — hash map + doubly linked list

Design questions don't share a loop. The reusable piece: a **doubly linked list with two sentinels** (`O(1)` remove and add-to-front) plus a **dict from key to node**. `OrderedDict` with `move_to_end` / `popitem(last=False)` is the 5-line version.

In [23]:
class Node:
    """Doubly linked node."""

    def __init__(self, key=0, val=0):
        self.key, self.val = key, val
        self.prev = self.next = None


class LRUCache:
    """Most recent right after head, least recent right before tail."""

    def __init__(self, capacity):
        self.capacity, self.map = capacity, {}
        self.head, self.tail = Node(), Node()
        self.head.next, self.tail.prev = self.tail, self.head

    def _remove(self, node):
        node.prev.next, node.next.prev = node.next, node.prev

    def _add_front(self, node):
        node.prev, node.next = self.head, self.head.next
        self.head.next.prev = node
        self.head.next = node

    def get(self, key):
        if key not in self.map:
            return -1
        node = self.map[key]
        self._remove(node)
        self._add_front(node)
        return node.val

    def put(self, key, val):
        if key in self.map:
            self._remove(self.map[key])
        node = self.map[key] = Node(key, val)
        self._add_front(node)
        if len(self.map) > self.capacity:
            lru = self.tail.prev
            self._remove(lru)
            del self.map[lru.key]


cache_ = LRUCache(2)
cache_.put(1, 1)
cache_.put(2, 2)
print(cache_.get(1), end=" ")
cache_.put(3, 3)
print(cache_.get(2), cache_.get(3))

1 -1 3


## 4.3 Trees, Tries and Heaps

In [24]:
class TreeNode:
    """Binary tree node."""

    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    """Tree from a LeetCode level-order list (None = missing child)."""
    if not values:
        return None
    root = TreeNode(values[0])
    queue, i = deque([root]), 1
    while queue and i < len(values):
        node = queue.popleft()
        for side in ("left", "right"):
            if i < len(values) and values[i] is not None:
                child = TreeNode(values[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root


print(build_tree([3, 9, 20, None, None, 15, 7]).right.left.val)

15


### 4.3.1 Tree bottom-up recursion · 🟢

**Practice:** 04 §1.1 · **Template:** `dfs(node)`: base → recurse both → update best → return

Write in words what `dfs` returns; that sentence is the 1–3 lines you change.

| Question | Returns | `best` update |
|---|---|---|
| 104 Maximum Depth | `1 + max(l, r)` | — |
| 543 Diameter | `1 + max(l, r)` | `l + r` |
| 110 Balanced | height, or −1 if `abs(l - r) > 1` | — |
| 124 Max Path Sum | `val + max(l, r, 0)` | `val + max(l, 0) + max(r, 0)` |
| 236 Lowest Common Ancestor | node if found below, else `None`; both sides found → this node | — |
| 337 House Robber III | pair `(rob, skip)` | — |
| 100 Same · 101 Symmetric · 572 Subtree | `dfs(a, b)` on two nodes at once | — |
| 226 Invert | swap children, return node | — |

In [25]:
def diameter(root):
    """Template: return one branch upward, record the bend in best."""
    best = 0

    def dfs(node):
        nonlocal best
        if not node:
            # CHANGE: value for an empty subtree
            return 0
        left, right = dfs(node.left), dfs(node.right)
        # CHANGE: answer that bends through this node
        best = max(best, left + right)
        # CHANGE: what the parent may use (one branch only)
        return 1 + max(left, right)

    dfs(root)
    return best


def lca(root, p, q):
    """236: same skeleton, no best."""
    if not root or root.val in (p, q):
        return root
    left, right = lca(root.left, p, q), lca(root.right, p, q)
    return root if left and right else left or right


def rob_tree(root):
    """337: return a pair (rob this node, skip this node)."""

    def dfs(node):
        if not node:
            return 0, 0
        lr, ls = dfs(node.left)
        rr, rs = dfs(node.right)
        return node.val + ls + rs, max(lr, ls) + max(rr, rs)

    return max(dfs(root))


tree = build_tree([3, 5, 1, 6, 2, 0, 8])
print(
    diameter(tree),
    lca(tree, 6, 2).val,
    rob_tree(build_tree([3, 2, 3, None, 3, None, 1])),
)

4 5 7


### 4.3.2 Tree top-down paths · 🟢

**Practice:** 04 §1.2 · **Template:** `dfs(node, state)`: update state, check, pass down

| Question | State passed down | Record when |
|---|---|---|
| 113 Path Sum II | path list + remaining sum (pop after) | leaf with remaining 0 |
| 437 Path Sum III | running prefix + Counter of prefixes on the path | every node: add `seen[prefix - target]` |
| 129 Sum Root to Leaf | `number * 10 + val` | leaf |
| 1448 Count Good Nodes | max value so far | `val >= max_so_far` |
| 1372 Longest ZigZag | `(went_left, length)` | every node |

In [26]:
def good_nodes(root):
    """Template: state flows from root to leaves."""

    def dfs(node, best_so_far):
        if not node:
            return 0
        # CHANGE: does this node count?
        here = 1 if node.val >= best_so_far else 0
        # CHANGE: new state for the children
        state = max(best_so_far, node.val)
        return here + dfs(node.left, state) + dfs(node.right, state)

    return dfs(root, float("-inf"))


def path_sum_count(root, target):
    """437: prefix sums along the current root-to-node path."""
    seen = Counter({0: 1})

    def dfs(node, prefix):
        if not node:
            return 0
        prefix += node.val
        total = seen[prefix - target]
        seen[prefix] += 1
        total += dfs(node.left, prefix) + dfs(node.right, prefix)
        # undo: this prefix is not on the path of other branches
        seen[prefix] -= 1
        return total

    return dfs(root, 0)


print(good_nodes(build_tree([3, 1, 4, 3, None, 1, 5])))
print(path_sum_count(build_tree([10, 5, -3, 3, 2, None, 11, 3, -2, None, 1]), 8))

4
3


### 4.3.3 Tree construction and serialisation · 🔴

**Practice:** 04 §1.3 · **Template:** none — five separate ideas

No shared template. The ideas to remember:

| Question | Idea |
|---|---|
| 105 Pre + Inorder · 106 In + Postorder | root comes from pre (front) / post (back); its inorder index splits left and right |
| 297 Serialize / Deserialize | preorder with `#` for `None`; rebuild by reading the same order from an iterator |
| 114 Flatten to Linked List | reverse preorder (right, left, node) with a `prev` pointer |
| 863 All Nodes Distance K | add parent links, then BFS from the target like a graph |
| 427 Construct Quad Tree | split into 4, merge if all four are equal leaves |

In [27]:
def build_pre_in(preorder, inorder):
    """105: next preorder value is the root; inorder splits it."""
    where = {v: i for i, v in enumerate(inorder)}
    values = iter(preorder)

    def build(lo, hi):
        if lo > hi:
            return None
        node = TreeNode(next(values))
        mid = where[node.val]
        node.left = build(lo, mid - 1)
        node.right = build(mid + 1, hi)
        return node

    return build(0, len(inorder) - 1)


def serialize(root):
    """297: preorder with '#' for empty."""
    if not root:
        return "#"
    return f"{root.val},{serialize(root.left)},{serialize(root.right)}"


def deserialize(data):
    """Read the tokens back in the same preorder."""
    tokens = iter(data.split(","))

    def build():
        token = next(tokens)
        if token == "#":
            return None
        node = TreeNode(int(token))
        node.left, node.right = build(), build()
        return node

    return build()


def distance_k(root, target, k):
    """863: parent links turn the tree into a graph, then BFS."""
    parent, stack, start = {root: None}, [root], None
    while stack:
        node = stack.pop()
        if node.val == target:
            start = node
        for child in (node.left, node.right):
            if child:
                parent[child] = node
                stack.append(child)
    seen, frontier = {start}, [start]
    for _ in range(k):
        nxt = []
        for node in frontier:
            for nb in (node.left, node.right, parent[node]):
                if nb and nb not in seen:
                    seen.add(nb)
                    nxt.append(nb)
        frontier = nxt
    return sorted(node.val for node in frontier)


tree = build_pre_in([3, 9, 20, 15, 7], [9, 3, 15, 20, 7])
print(serialize(tree), serialize(deserialize(serialize(tree))) == serialize(tree))
print(distance_k(build_tree([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4]), 5, 2))

3,9,#,#,20,15,#,#,7,#,# True
[1, 4, 7]


### 4.3.4 Tree level-order BFS · 🟢

**Practice:** 04 §2.1 · **Template:** `for _ in range(len(queue))` = one level

| Question | What you change |
|---|---|
| 102 Level Order | append the level's values |
| 199 Right Side View | keep the last value of each level |
| 103 Zigzag Level Order | reverse every other level |
| 1161 Max Level Sum | level sum, keep the best level number |
| 662 Max Width | queue holds `(node, index)`; children `2i`, `2i + 1`; width = last − first + 1 |
| 117 Next Right Pointers II | link each node to the next one in its level |
| 314 Vertical Order | queue holds `(node, column)`; group values by column |

In [28]:
def level_order(root):
    """Template: one pass of the for-loop = one level."""
    if not root:
        return []
    out, queue = [], deque([root])
    while queue:
        level = []
        for _ in range(len(queue)):
            node = queue.popleft()
            level.append(node.val)
            for child in (node.left, node.right):
                if child:
                    queue.append(child)
        # CHANGE: what you keep per level (level, level[-1], sum, reversed...)
        out.append(level)
    return out


def max_width(root):
    """662: carry a position index with each node."""
    best, queue = 0, deque([(root, 0)])
    while queue:
        first, last = queue[0][1], queue[-1][1]
        best = max(best, last - first + 1)
        for _ in range(len(queue)):
            node, i = queue.popleft()
            if node.left:
                queue.append((node.left, 2 * (i - first)))
            if node.right:
                queue.append((node.right, 2 * (i - first) + 1))
    return best


print(
    level_order(build_tree([3, 9, 20, None, None, 15, 7])),
    max_width(build_tree([1, 3, 2, 5, 3, None, 9])),
)

[[3], [9, 20], [15, 7]] 4


### 4.3.5 BST in-order and bounds · 🟡

**Practice:** 04 §3.1 · **Template:** in-order walk (sorted order) · walk down by comparing

The condition: **do you need the values in sorted order, or one spot in the tree?**

| Question | Template | What you change |
|---|---|---|
| 230 Kth Smallest · 173 BST Iterator | in-order (stack) | stop at the k-th pop / pop on `next()` |
| 98 Validate BST | bounds | pass `(low, high)` down; every node must sit strictly inside |
| 235 LCA of a BST | walk down | both smaller → left, both bigger → right, else here |
| 285 Inorder Successor | walk down | remember the last node where you turned left |
| 701 Insert into a BST | walk down | attach at the first empty spot |
| 450 Delete Node | walk down | 0/1 child → splice; 2 children → copy successor, delete it on the right |

In [29]:
def inorder(root):
    """In-order with an explicit stack: yields values in sorted order."""
    stack, node = [], root
    while stack or node:
        while node:
            stack.append(node)
            node = node.left
        node = stack.pop()
        yield node.val
        node = node.right


def kth_smallest(root, k):
    """230: the k-th value of the in-order walk."""
    for i, val in enumerate(inorder(root), 1):
        if i == k:
            return val


def is_valid_bst(node, low=float("-inf"), high=float("inf")):
    """98: bounds shrink as you go down."""
    if not node:
        return True
    if not low < node.val < high:
        return False
    return is_valid_bst(node.left, low, node.val) and is_valid_bst(
        node.right, node.val, high
    )


def lca_bst(root, p, q):
    """235: walk down until p and q split."""
    node = root
    while node:
        if p < node.val and q < node.val:
            node = node.left
        elif p > node.val and q > node.val:
            node = node.right
        else:
            return node


bst = build_tree([6, 2, 8, 0, 4, 7, 9, None, None, 3, 5])
print(
    list(inorder(bst)), kth_smallest(bst, 3), is_valid_bst(bst), lca_bst(bst, 3, 5).val
)

[0, 2, 3, 4, 5, 6, 7, 8, 9] 3 True 4


### 4.3.6 Trie (prefix tree) · 🟢

**Practice:** 04 §4.1 · **Template:** nested dicts + an end marker

| Question | What you change |
|---|---|
| 208 Implement Trie | insert / search / starts_with (the template) |
| 211 Add and Search Words | `.` → try every child (DFS) |
| 1268 Search Suggestions | walk the prefix, then collect up to 3 words in sorted order |
| 212 Word Search II | DFS the grid while walking the trie; prune nodes once their word is found |

In [30]:
END = "$"


def trie_insert(root, word):
    """Template: nested dicts, END marks a full word."""
    node = root
    for ch in word:
        node = node.setdefault(ch, {})
    node[END] = word


def trie_find(root, prefix):
    """Node at the end of prefix, or None."""
    node = root
    for ch in prefix:
        if ch not in node:
            return None
        node = node[ch]
    return node


def wildcard_search(node, word, i=0):
    """211: '.' tries every child."""
    if i == len(word):
        return END in node
    ch = word[i]
    if ch == ".":
        return any(
            wildcard_search(child, word, i + 1)
            for key, child in node.items()
            if key != END
        )
    return ch in node and wildcard_search(node[ch], word, i + 1)


root = {}
for w in ["apple", "app", "bad", "dad"]:
    trie_insert(root, w)
node = trie_find(root, "app")
print(
    node is not None and END in node,
    trie_find(root, "ap") is not None,
    wildcard_search(root, ".ad"),
)

True True True


### 4.3.7 Heap top-k and k-th element · 🟢

**Practice:** 04 §5.1 · **Template:** size-k min-heap · quickselect

| Question | What you change |
|---|---|
| 703 Kth Largest in a Stream · 215 Kth Largest | keep k largest; `heap[0]` is the answer |
| 215 (O(n) average) | quickselect: 3-way split around a random pivot, recurse into the side holding the k-th |
| 973 K Closest Points | keep k smallest → push `(-dist, point)` |
| 692 Top K Frequent Words | key = `(-count, word)` with `nsmallest` (tie-break by word) |
| 1046 Last Stone Weight | max-heap simulation (negate) |
| 373 K Pairs with Smallest Sums | frontier heap: pop `(i, j)`, push `(i, j + 1)` |

In [31]:
def k_largest(nums, k):
    """Template: the heap keeps the k best seen so far; root = the weakest of them."""
    heap = []
    for x in nums:
        # CHANGE: the key (x, -dist, (count, word) ...)
        heapq.heappush(heap, x)
        if len(heap) > k:
            heapq.heappop(heap)
    return heap[0]


def k_closest(points, k):
    """973: keep k smallest distances -> negate into a max-heap."""
    heap = []
    for x, y in points:
        heapq.heappush(heap, (-(x * x + y * y), x, y))
        if len(heap) > k:
            heapq.heappop(heap)
    return sorted([x, y] for _, x, y in heap)


def k_smallest_pairs(a, b, k):
    """373: frontier heap; each pop unlocks its right neighbour."""
    heap = [(a[i] + b[0], i, 0) for i in range(min(k, len(a)))]
    out = []
    while heap and len(out) < k:
        _, i, j = heapq.heappop(heap)
        out.append([a[i], b[j]])
        if j + 1 < len(b):
            heapq.heappush(heap, (a[i] + b[j + 1], i, j + 1))
    return out


def quickselect(nums, k):
    """k-th largest in O(n) average: split, then keep only the side that holds it."""
    pivot = random.choice(nums)
    bigger = [x for x in nums if x > pivot]
    equal = [x for x in nums if x == pivot]
    if k <= len(bigger):
        return quickselect(bigger, k)
    if k <= len(bigger) + len(equal):
        return pivot
    return quickselect([x for x in nums if x < pivot], k - len(bigger) - len(equal))


print(quickselect([3, 2, 3, 1, 2, 4, 5, 5, 6], 4))
print(k_largest([3, 2, 1, 5, 6, 4], 2), k_closest([[3, 3], [5, -1], [-2, 4]], 2))
print(k_smallest_pairs([1, 7, 11], [2, 4, 6], 3))

4
5 [[-2, 4], [3, 3]]
[[1, 2], [1, 4], [1, 6]]


### 4.3.8 Heap scheduling and simulation · 🟡

**Practice:** 04 §5.2 · **Template:** sort by one key + heap on the other · two heaps · greedy max

Three shapes; the condition is what the heap is protecting.

| Shape | Questions | What you change |
|---|---|---|
| Greedy: always take the most frequent / largest | 621, 767, 355 (merge newest tweets) | what "used" means (cooldown queue, previous char) |
| Two heaps (low max-heap, high min-heap) | 295 | — (the template) |
| Sort by one key, heap holds the best of the other | 1834 (sort by time, heap by duration), 2542 (sort by multiplier, heap of top-k sums), 1642 (heap of ladder climbs) | the sort key and when to pop |

In [32]:
class MedianFinder:
    """295: low = max-heap (negated), high = min-heap; sizes differ by <= 1."""

    def __init__(self):
        self.low, self.high = [], []

    def add(self, x):
        heapq.heappush(self.low, -x)
        heapq.heappush(self.high, -heapq.heappop(self.low))
        if len(self.high) > len(self.low):
            heapq.heappush(self.low, -heapq.heappop(self.high))

    def median(self):
        if len(self.low) > len(self.high):
            return -self.low[0]
        return (-self.low[0] + self.high[0]) / 2


def reorganize(s):
    """767: take the most frequent char that isn't the previous one."""
    heap = [(-c, ch) for ch, c in Counter(s).items()]
    heapq.heapify(heap)
    out, held = [], None
    while heap:
        c, ch = heapq.heappop(heap)
        out.append(ch)
        # release the char held back last round
        if held:
            heapq.heappush(heap, held)
        held = (c + 1, ch) if c + 1 < 0 else None
    return "".join(out) if not held else ""


def max_score(nums1, nums2, k):
    """2542: sort by multiplier desc; heap keeps the k largest nums1."""
    heap, total, best = [], 0, 0
    for mult, x in sorted(zip(nums2, nums1), reverse=True):
        heapq.heappush(heap, x)
        total += x
        if len(heap) > k:
            total -= heapq.heappop(heap)
        if len(heap) == k:
            best = max(best, total * mult)
    return best


mf = MedianFinder()
for x in [1, 2, 3]:
    mf.add(x)
print(mf.median(), reorganize("aab"), max_score([1, 3, 3, 2], [2, 1, 3, 4], 3))

2 aba 12


## 4.4 Graphs and Backtracking

### 4.4.1 Grid connected regions (DFS) · 🟢

**Practice:** 05 §1.1 · **Template:** for each start cell: flood its region and mark it

| Question | What you change |
|---|---|
| 200 Number of Islands | count flood starts |
| 695 Max Area of Island | flood returns the cell count |
| 733 Flood Fill | one start; repaint instead of marking |
| 130 Surrounded Regions | flood from border `O`s first (safe), flip the rest |
| 417 Pacific Atlantic | flood uphill from each ocean's border; answer = cells in both |

In [33]:
DIRS = [(1, 0), (-1, 0), (0, 1), (0, -1)]


def island_areas(grid):
    """Template: flood every unvisited land cell (stack = no recursion limit)."""
    rows, cols = len(grid), len(grid[0])
    seen, areas = set(), []
    for r in range(rows):
        for c in range(cols):
            # CHANGE: what counts as a start cell
            if grid[r][c] != 1 or (r, c) in seen:
                continue
            seen.add((r, c))
            stack, area = [(r, c)], 0
            while stack:
                cr, cc = stack.pop()
                area += 1
                for dr, dc in DIRS:
                    nr, nc = cr + dr, cc + dc
                    # CHANGE: which neighbour you may step to
                    if (
                        0 <= nr < rows
                        and 0 <= nc < cols
                        and grid[nr][nc] == 1
                        and (nr, nc) not in seen
                    ):
                        seen.add((nr, nc))
                        stack.append((nr, nc))
            areas.append(area)
    return areas


grid = [[1, 1, 0, 0], [1, 0, 0, 1], [0, 0, 1, 1]]
# 200 = how many, 695 = the biggest
print(len(island_areas(grid)), max(island_areas(grid)))

2 3


### 4.4.2 Grid shortest paths (BFS) · 🟢

**Practice:** 05 §1.2 · **Template:** multi-source BFS, one ring per step

| Question | Sources | What you change |
|---|---|---|
| 994 Rotting Oranges | all rotten | count rings; fresh left → −1 |
| 286 Walls and Gates | all gates | write the distance into rooms |
| 542 01 Matrix | all zeros | write the distance into ones |
| 1091 Shortest Path Binary Matrix | top-left | 8 directions; stop at bottom-right |
| 1926 Nearest Exit | entrance | stop at the first border cell that isn't the entrance |
| 934 Shortest Bridge | flood island 1 first (DFS) | BFS until you touch island 2 |

In [34]:
def grid_bfs(grid, sources, passable):
    """Template: distance from the nearest source to every reachable cell."""
    rows, cols = len(grid), len(grid[0])
    dist = {cell: 0 for cell in sources}
    queue = deque(sources)
    while queue:
        r, c = queue.popleft()
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if (
                0 <= nr < rows
                and 0 <= nc < cols
                and (nr, nc) not in dist
                and passable(grid[nr][nc])
            ):
                dist[(nr, nc)] = dist[(r, c)] + 1
                queue.append((nr, nc))
    return dist


def oranges_rotting(grid):
    """994: sources = rotten, passable = fresh."""
    cells = [(r, c) for r in range(len(grid)) for c in range(len(grid[0]))]
    # CHANGE: sources and passable test
    dist = grid_bfs(grid, [x for x in cells if grid[x[0]][x[1]] == 2], lambda v: v == 1)
    fresh = [x for x in cells if grid[x[0]][x[1]] == 1]
    if any(x not in dist for x in fresh):
        return -1
    return max(dist.values(), default=0)


print(oranges_rotting([[2, 1, 1], [1, 1, 0], [0, 1, 1]]))

4


### 4.4.3 Adjacency-list traversal · 🟡

**Practice:** 05 §2.1 · **Template:** build graph → visited set → DFS/BFS

Same traversal; the condition is **how to build the graph** and **what you check on each edge**.

| Question | Graph | On each visit / edge |
|---|---|---|
| 133 Clone Graph | given nodes | `old → clone` map doubles as visited |
| 547 Provinces · 323 Components | undirected | count DFS starts |
| 841 Keys and Rooms | directed | reach everything from 0? |
| 261 Graph Valid Tree | undirected | `n - 1` edges and connected |
| 785 Is Graph Bipartite | undirected | colour 0/1; same colour on an edge → `False` |
| 1466 Reorder Routes | store both directions, cost 1 on original edges | sum costs from 0 |
| 399 Evaluate Division | weighted: `a → b = v`, `b → a = 1/v` | multiply weights along the path |

In [35]:
def build_graph(edges, directed=False):
    """Adjacency list from an edge list."""
    graph = defaultdict(list)
    for u, v in edges:
        graph[u].append(v)
        if not directed:
            graph[v].append(u)
    return graph


def count_components(n, edges):
    """Template: count how many times a fresh DFS starts."""
    graph, seen, count = build_graph(edges), set(), 0
    for start in range(n):
        if start in seen:
            continue
        count += 1
        seen.add(start)
        stack = [start]
        while stack:
            u = stack.pop()
            for v in graph[u]:
                if v not in seen:
                    seen.add(v)
                    stack.append(v)
    return count


def is_bipartite(graph):
    """785: same loop, a colour instead of a seen-set."""
    colour = {}
    for start in range(len(graph)):
        if start in colour:
            continue
        colour[start] = 0
        stack = [start]
        while stack:
            u = stack.pop()
            for v in graph[u]:
                if v not in colour:
                    colour[v] = 1 - colour[u]
                    stack.append(v)
                elif colour[v] == colour[u]:
                    return False
    return True


print(
    count_components(5, [(0, 1), (1, 2), (3, 4)]),
    is_bipartite([[1, 3], [0, 2], [1, 3], [0, 2]]),
)

2 True


### 4.4.4 BFS over state spaces · 🟢

**Practice:** 05 §2.2 · **Template:** BFS where `neighbours(state)` generates the edges

Each state is a node; you never build the graph, you generate neighbours.

| Question | `neighbours(state)` |
|---|---|
| 127 Word Ladder · 433 Genetic Mutation | change one letter; must be in the word bank |
| 752 Open the Lock | turn one wheel ±1; dead ends are pre-marked visited |
| 909 Snakes and Ladders | dice 1–6, then follow a snake/ladder |

In [36]:
def min_steps(start, goal, neighbours, blocked=()):
    """Template: fewest moves from start to goal (-1 if unreachable)."""
    if start in blocked:
        return -1
    seen, queue = {start, *blocked}, deque([(start, 0)])
    while queue:
        state, steps = queue.popleft()
        if state == goal:
            return steps
        for nxt in neighbours(state):
            if nxt not in seen:
                seen.add(nxt)
                queue.append((nxt, steps + 1))
    return -1


def lock_moves(code):
    """752: CHANGE only this generator."""
    for i in range(4):
        d = int(code[i])
        for turn in (-1, 1):
            yield code[:i] + str((d + turn) % 10) + code[i + 1 :]


print(
    min_steps(
        "0000", "0202", lock_moves, blocked={"0201", "0101", "0102", "1212", "2002"}
    )
)

6


### 4.4.5 Topological sort · 🟢

**Practice:** 05 §3.1 · **Template:** Kahn: queue of in-degree 0, peel, decrement

| Question | What you change |
|---|---|
| 207 Course Schedule | return `len(order) == n` |
| 210 Course Schedule II | return `order` (or `[]` on a cycle) |
| 269 Alien Dictionary | edges = first differing letter of adjacent words; longer word before its own prefix → invalid |
| 310 Minimum Height Trees | undirected: peel leaves (degree 1) ring by ring; last 1–2 remain |

In [37]:
def topo_order(n, edges):
    """Template: Kahn. Shorter than n means a cycle."""
    graph, indeg = defaultdict(list), [0] * n
    # CHANGE: how the edges are produced
    for u, v in edges:
        graph[u].append(v)
        indeg[v] += 1
    queue = deque(i for i in range(n) if indeg[i] == 0)
    order = []
    while queue:
        u = queue.popleft()
        order.append(u)
        for v in graph[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                queue.append(v)
    # CHANGE: what you return
    return order


def min_height_roots(n, edges):
    """310: peel leaves ring by ring."""
    graph = defaultdict(set)
    for u, v in edges:
        graph[u].add(v)
        graph[v].add(u)
    leaves, remaining = [i for i in range(n) if len(graph[i]) <= 1], n
    while remaining > 2:
        remaining -= len(leaves)
        nxt = []
        for leaf in leaves:
            nb = graph[leaf].pop()
            graph[nb].discard(leaf)
            if len(graph[nb]) == 1:
                nxt.append(nb)
        leaves = nxt
    return sorted(leaves)


print(
    topo_order(4, [(1, 0), (2, 0), (3, 1), (3, 2)]),
    min_height_roots(6, [(3, 0), (3, 1), (3, 2), (3, 4), (5, 4)]),
)

[3, 1, 2, 0] [3, 4]


### 4.4.6 Union-Find · 🟢

**Practice:** 05 §4.1 · **Template:** `UnionFind` class (find with path halving, union by size)

| Question | What you change |
|---|---|
| 684 Redundant Connection | first edge whose `union` returns `False` |
| 721 Accounts Merge | union every email of an account with its first email; group by root |
| 547 · 323 · 261 (alternative) | components = `n - successful unions` |

In [38]:
class UnionFind:
    """Disjoint sets with path halving and union by size."""

    def __init__(self, n):
        self.parent = list(range(n))
        self.size = [1] * n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        """Join the sets of a and b; False if they were already joined."""
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        if self.size[ra] < self.size[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        self.size[ra] += self.size[rb]
        return True


def redundant_edge(edges):
    """684: the edge that closes a cycle."""
    uf = UnionFind(len(edges) + 1)
    for u, v in edges:
        # CHANGE: what you do with the union result
        if not uf.union(u, v):
            return [u, v]


print(redundant_edge([[1, 2], [1, 3], [2, 3]]))

[2, 3]


### 4.4.7 Dijkstra and variants · 🟡

**Practice:** 05 §5.1 · **Template:** Dijkstra · Bellman–Ford (k rounds) · Prim / Kruskal

The condition: **how does a path cost combine, and is there a limit on edges?**

| Question | Choice | What you change |
|---|---|---|
| 743 Network Delay Time | Dijkstra | answer = `max(dist)` |
| 1631 Path With Minimum Effort | Dijkstra on the grid | path cost = `max(d, edge)` instead of `d + edge` |
| 787 Cheapest Flights Within K Stops | Bellman–Ford | `k + 1` rounds, relax from a copy of last round |
| 1584 Min Cost to Connect Points | Prim (MST) | heap holds edge weights, not path totals |
| 1584 (alternative) | Kruskal (MST) | sort all edges, `union` cheapest first until n − 1 joined (better for sparse edge lists) |

In [39]:
def dijkstra(graph, src):
    """Shortest distances; graph[u] = [(v, w)], w >= 0."""
    dist, heap = {src: 0}, [(0, src)]
    while heap:
        d, u = heapq.heappop(heap)
        # skip stale entries
        if d > dist[u]:
            continue
        for v, w in graph[u]:
            # CHANGE: how the path cost combines (1631: max(d, w))
            nd = d + w
            if nd < dist.get(v, float("inf")):
                dist[v] = nd
                heapq.heappush(heap, (nd, v))
    return dist


def cheapest_with_stops(n, flights, src, dst, k):
    """787: Bellman-Ford limited to k + 1 edges."""
    cost = [float("inf")] * n
    cost[src] = 0
    for _ in range(k + 1):
        # relax from last round's values only
        nxt = cost.copy()
        for u, v, w in flights:
            nxt[v] = min(nxt[v], cost[u] + w)
        cost = nxt
    return -1 if cost[dst] == float("inf") else cost[dst]


def prim(points):
    """1584: grow the tree with the cheapest edge out of it."""
    n, total, used = len(points), 0, set()
    heap = [(0, 0)]
    while len(used) < n:
        w, u = heapq.heappop(heap)
        if u in used:
            continue
        used.add(u)
        total += w
        for v in range(n):
            if v not in used:
                d = abs(points[u][0] - points[v][0]) + abs(points[u][1] - points[v][1])
                heapq.heappush(heap, (d, v))
    return total


def kruskal(n, edges):
    """MST weight: cheapest edges first, skip any that would close a cycle."""
    uf, total, used = UnionFind(n), 0, 0
    for w, u, v in sorted(edges):
        if uf.union(u, v):
            total += w
            used += 1
            if used == n - 1:
                break
    return total


g = defaultdict(list, {2: [(1, 1), (3, 1)], 3: [(4, 1)]})
print(
    max(dijkstra(g, 2).values()),
    cheapest_with_stops(
        4, [[0, 1, 100], [1, 2, 100], [2, 0, 100], [1, 3, 600], [2, 3, 200]], 0, 3, 1
    ),
)
pts = [[0, 0], [2, 2], [3, 10], [5, 2], [7, 0]]
all_edges = [
    (abs(a[0] - b[0]) + abs(a[1] - b[1]), i, j)
    for i, a in enumerate(pts)
    for j, b in enumerate(pts)
    if i < j
]
print(prim(pts), kruskal(len(pts), all_edges))

2 700
20 20


### 4.4.8 Subsets, combinations, permutations · 🟢

**Practice:** 05 §6.1 · **Template:** choose → explore → un-choose, from `start`

Everything is the subsets template with 1–3 lines changed.

| Question | What you change |
|---|---|
| 78 Subsets | record every path; recurse `i + 1` |
| 90 Subsets II | sort; skip `i > start and nums[i] == nums[i - 1]` |
| 77 Combinations | record only when `len(path) == k` |
| 39 Combination Sum | recurse `i` (reuse); stop when remaining < 0 |
| 40 Combination Sum II | recurse `i + 1` + the 90 skip |
| 216 Combination Sum III | numbers 1–9; record at length k and sum n |
| 46 Permutations | loop from 0 with a `used` array |
| 47 Permutations II | `used` + skip if the equal previous one is unused |
| 17 Letter Combinations | one position per digit; loop over its letters |
| 22 Generate Parentheses | add `(` if `open < n`, `)` if `close < open` |

In [40]:
def subsets(nums):
    """Template: choose, explore, un-choose."""
    out, path = [], []

    def backtrack(start):
        # CHANGE: when to record (always / len == k / remaining == 0)
        out.append(path.copy())
        for i in range(start, len(nums)):
            # CHANGE: skip duplicates (sorted): if i > start and nums[i] == nums[i - 1]
            path.append(nums[i])
            # CHANGE: i + 1 (each once) or i (reuse allowed)
            backtrack(i + 1)
            path.pop()

    backtrack(0)
    return out


def combination_sum(candidates, target):
    """39: three lines changed (record rule, prune, recurse with i)."""
    out, path = [], []

    def backtrack(start, remaining):
        if remaining == 0:
            out.append(path.copy())
            return
        for i in range(start, len(candidates)):
            if candidates[i] > remaining:
                continue
            path.append(candidates[i])
            backtrack(i, remaining - candidates[i])
            path.pop()

    backtrack(0, target)
    return out


def permutations(nums):
    """46: loop from 0, skip used."""
    out, path, used = [], [], [False] * len(nums)

    def backtrack():
        if len(path) == len(nums):
            out.append(path.copy())
            return
        for i in range(len(nums)):
            if used[i]:
                continue
            used[i] = True
            path.append(nums[i])
            backtrack()
            path.pop()
            used[i] = False

    backtrack()
    return out


print(subsets([1, 2, 3]))
print(combination_sum([2, 3, 6, 7], 7), permutations([1, 2, 3])[:3])

[[], [1], [1, 2], [1, 2, 3], [1, 3], [2], [2, 3], [3]]
[[2, 2, 3], [7]] [[1, 2, 3], [1, 3, 2], [2, 1, 3]]


### 4.4.9 Backtracking with constraints · 🟡

**Practice:** 05 §6.2 · **Template:** backtrack + `is_valid` check before each choice

Same choose / explore / un-choose; the condition is **what a choice is** and **when it's legal**.

| Question | A choice is | Legal when |
|---|---|---|
| 79 Word Search | step to a neighbour cell (mark, recurse, unmark) | in grid, unvisited, matches next letter |
| 131 Palindrome Partitioning | cut `s[start:end]` | the piece is a palindrome |
| 93 Restore IP Addresses | cut 1–3 digits | 0–255, no leading zero, 4 parts total |
| 51 N-Queens | a column for this row | column, `r - c`, `r + c` all free |
| 698 Partition K Equal · 473 Matchsticks | a bucket for this number (sorted desc) | fits; skip buckets with the same load |

In [41]:
def word_exists(board, word):
    """79: grid backtracking - mark, explore 4 ways, unmark."""
    rows, cols = len(board), len(board[0])

    def backtrack(r, c, i):
        if i == len(word):
            return True
        if not (0 <= r < rows and 0 <= c < cols) or board[r][c] != word[i]:
            return False
        board[r][c] = "#"
        found = any(backtrack(r + dr, c + dc, i + 1) for dr, dc in DIRS)
        board[r][c] = word[i]
        return found

    return any(backtrack(r, c, 0) for r in range(rows) for c in range(cols))


def palindrome_partitions(s):
    """131: the choice is where to cut; legal if the piece is a palindrome."""
    out, path = [], []

    def backtrack(start):
        if start == len(s):
            out.append(path.copy())
            return
        for end in range(start + 1, len(s) + 1):
            piece = s[start:end]
            # CHANGE: the legality test
            if piece == piece[::-1]:
                path.append(piece)
                backtrack(end)
                path.pop()

    backtrack(0)
    return out


def n_queens(n):
    """51: count boards; three sets track attacked columns and diagonals."""
    cols, diag, anti = set(), set(), set()

    def backtrack(r):
        if r == n:
            return 1
        total = 0
        for c in range(n):
            if c in cols or r - c in diag or r + c in anti:
                continue
            cols.add(c), diag.add(r - c), anti.add(r + c)
            total += backtrack(r + 1)
            cols.remove(c), diag.remove(r - c), anti.remove(r + c)
        return total

    return backtrack(0)


board = [["A", "B", "C", "E"], ["S", "F", "C", "S"], ["A", "D", "E", "E"]]
print(word_exists(board, "ABCCED"), palindrome_partitions("aab"), n_queens(6))

True [['a', 'a', 'b'], ['aa', 'b']] 4


## 4.5 Dynamic Programming

### 4.5.1 Fibonacci-style recurrences · 🟢

**Practice:** 06 §1.1 · **Template:** roll two variables: `prev2, prev1 = prev1, new`

| Question | The `new = …` line |
|---|---|
| 70 Climbing Stairs | `prev1 + prev2` |
| 746 Min Cost Climbing Stairs | `cost[i] + min(prev1, prev2)` |
| 198 House Robber | `max(prev1, prev2 + x)` |
| 213 House Robber II | run 198 on `nums[1:]` and `nums[:-1]` |
| 740 Delete and Earn | bucket points by value, then 198 |
| 91 Decode Ways | `prev1` if one digit valid + `prev2` if two digits valid |
| 790 Domino and Tromino | `dp[n] = 2·dp[n-1] + dp[n-3]` (keep three) |
| 983 Min Cost Tickets | `dp[day]` = min over the 1/7/30-day passes (a full array) |

In [42]:
def rob(nums):
    """Template: each step only needs the last two answers."""
    prev2 = prev1 = 0
    for x in nums:
        # CHANGE: the recurrence
        new = max(prev1, prev2 + x)
        prev2, prev1 = prev1, new
    return prev1


def decode_ways(s):
    """91: same roll, the new value adds the valid one- and two-digit options."""
    prev2, prev1 = 0, 1
    for i, ch in enumerate(s):
        new = prev1 if ch != "0" else 0
        if i and 10 <= int(s[i - 1 : i + 1]) <= 26:
            new += prev2
        prev2, prev1 = prev1, new
    return prev1


print(rob([2, 7, 9, 3, 1]), decode_ways("226"), decode_ways("06"))

12 3 0


### 4.5.2 Kadane and subarray DP · 🟢

**Practice:** 06 §1.2 · **Template:** `cur = max(x, cur + x)`; best = max(best, cur)

| Question | What you change |
|---|---|
| 53 Maximum Subarray | the template |
| 152 Maximum Product Subarray | track `cur_max` and `cur_min` (a negative flips them) |
| 918 Max Circular Subarray | `max(best, total - min_subarray)`, unless all values are negative |

In [43]:
def max_subarray(nums):
    """Template: extend the run or restart at x."""
    cur = best = nums[0]
    for x in nums[1:]:
        # CHANGE: the extend-or-restart rule
        cur = max(x, cur + x)
        best = max(best, cur)
    return best


def max_product(nums):
    """152: also keep the most negative product."""
    hi = lo = best = nums[0]
    for x in nums[1:]:
        hi, lo = max(x, hi * x, lo * x), min(x, hi * x, lo * x)
        best = max(best, hi)
    return best


def max_circular(nums):
    """918: wrap-around best = total - the worst middle run."""
    best = max_subarray(nums)
    worst = -max_subarray([-x for x in nums])
    return best if best < 0 else max(best, sum(nums) - worst)


print(
    max_subarray([-2, 1, -3, 4, -1, 2, 1, -5, 4]),
    max_product([2, 3, -2, 4]),
    max_circular([5, -3, 5]),
)

6 6 10


### 4.5.3 Word and partition DP · 🟡

**Practice:** 06 §1.3 · **Template:** `dp[i]` = combine over every last piece

The condition: **what is the last piece, and how do the options combine?**

| Question | Last piece | Combine |
|---|---|---|
| 139 Word Break | a dictionary word ending at `i` | `any` |
| 279 Perfect Squares | a square `≤ i` | `1 + min` |
| 343 Integer Break | first part `j` | `max(j · (i − j), j · dp[i − j])` |
| 96 Unique BSTs | the root | `sum(dp[left] · dp[right])` |

In [44]:
def word_break(s, words):
    """139: dp[i] = can s[:i] be split?"""
    dp = [True] + [False] * len(s)
    for i in range(1, len(s) + 1):
        dp[i] = any(dp[j] and s[j:i] in words for j in range(i))
    return dp[-1]


def num_squares(n):
    """279: dp[i] = fewest squares summing to i."""
    dp = [0] + [float("inf")] * n
    for i in range(1, n + 1):
        j = 1
        while j * j <= i:
            dp[i] = min(dp[i], 1 + dp[i - j * j])
            j += 1
    return dp[n]


def num_trees(n):
    """96: dp[i] = BSTs with i nodes; try every root."""
    dp = [1] + [0] * n
    for i in range(1, n + 1):
        dp[i] = sum(dp[root - 1] * dp[i - root] for root in range(1, i + 1))
    return dp[n]


print(word_break("leetcode", {"leet", "code"}), num_squares(12), num_trees(3))

True 3 5


### 4.5.4 Longest increasing subsequence · 🟢

**Practice:** 06 §1.4 · **Template:** `tails` + `bisect_left` (or O(n²) `dp[i]`)

One question, but the template returns in follow-ups (354 Russian Dolls: sort by width asc, height desc, then LIS on heights).

| Version | When |
|---|---|
| `tails` + binary search, O(n log n) | only the length is needed |
| `dp[i] = 1 + max(dp[j] for j < i if a[j] < a[i])`, O(n²) | you also need counts or the actual sequence |

In [45]:
def lis_length(nums):
    """tails[k] = smallest tail of an increasing run of length k + 1."""
    tails = []
    for x in nums:
        # CHANGE: bisect_right for non-decreasing
        i = bisect_left(tails, x)
        if i == len(tails):
            tails.append(x)
        else:
            tails[i] = x
    return len(tails)


def lis_quadratic(nums):
    """O(n^2): dp[i] = longest run ending at i."""
    dp = [1] * len(nums)
    for i in range(len(nums)):
        for j in range(i):
            if nums[j] < nums[i]:
                dp[i] = max(dp[i], dp[j] + 1)
    return max(dp)


print(lis_length([10, 9, 2, 5, 3, 7, 101, 18]), lis_quadratic([0, 1, 0, 3, 2, 3]))

4 4


### 4.5.5 Knapsack: subset sums and coins · 🟢

**Practice:** 06 §2.1 · **Template:** 1D `dp` over capacity

The famous 3-line switchboard:

| Question | Items | Combine | Loop |
|---|---|---|---|
| 322 Coin Change | unbounded | `min(dp[c], dp[c - x] + 1)`, start `inf` | capacity **up** |
| 518 Coin Change II | unbounded, combinations | `dp[c] += dp[c - x]` | **coins outer**, capacity up |
| 377 Combination Sum IV | unbounded, orderings | `dp[c] += dp[c - x]` | **capacity outer**, coins inner |
| 416 Partition Equal Subset | 0/1 | `dp[c] = dp[c] or dp[c - x]`, target = sum / 2 | capacity **down** |
| 494 Target Sum | 0/1 | `dp[c] += dp[c - x]`, target = (sum + t) / 2 | capacity down |
| 1049 Last Stone Weight II | 0/1 | reachable sums; best `≤ sum / 2` | capacity down |

In [46]:
def knapsack(items, capacity, reuse, combine, start, empty):
    """Template: dp[c] = best answer using capacity c."""
    dp = [empty] * (capacity + 1)
    dp[0] = start
    for x in items:
        # CHANGE: up = reuse allowed (unbounded); down = each item once (0/1)
        caps = range(x, capacity + 1) if reuse else range(capacity, x - 1, -1)
        for c in caps:
            # CHANGE: the combine rule
            dp[c] = combine(dp[c], dp[c - x])
    return dp[capacity]


# 322: fewest coins
print(knapsack([1, 2, 5], 11, True, lambda a, b: min(a, b + 1), 0, float("inf")))
# 518: number of combinations
print(knapsack([1, 2, 5], 5, True, lambda a, b: a + b, 1, 0))
# 416: can [1, 5, 11, 5] split into two equal halves?
print(knapsack([1, 5, 11, 5], 11, False, lambda a, b: a or b, True, False))


def combination_sum4(nums, target):
    """377: orderings count -> capacity is the OUTER loop."""
    dp = [1] + [0] * target
    for c in range(1, target + 1):
        dp[c] = sum(dp[c - x] for x in nums if x <= c)
    return dp[target]


print(combination_sum4([1, 2, 3], 4))

3
4
True
7


### 4.5.6 State machine DP (stocks) · 🟢

**Practice:** 06 §3.1 · **Template:** one variable per state, all updated each day

| Question | What you change |
|---|---|
| 122 Stock II | `hold = max(hold, free - p)`, `free = max(free, hold + p)` |
| 714 With Fee | subtract `fee` when selling |
| 309 With Cooldown | add a `cool` state: sell → `cool`, `cool` → `free` next day |

In [47]:
def max_profit(prices, fee=0):
    """Template: hold (own a share) / free (own nothing)."""
    hold, free = float("-inf"), 0
    for p in prices:
        # CHANGE: transitions (fee, extra states)
        hold, free = max(hold, free - p), max(free, hold + p - fee)
    return free


def max_profit_cooldown(prices):
    """309: selling sends you to 'cool' for one day."""
    hold, free, cool = float("-inf"), 0, 0
    for p in prices:
        hold, free, cool = max(hold, free - p), max(free, cool), hold + p
    return max(free, cool)


print(
    max_profit([7, 1, 5, 3, 6, 4]),
    max_profit([1, 3, 2, 8, 4, 9], 2),
    max_profit_cooldown([1, 2, 3, 0, 2]),
)

7 8 3


### 4.5.7 Grid DP · 🟢

**Practice:** 06 §4.1 · **Template:** `dp[r][c]` from up and left (one rolling row)

| Question | The combine line |
|---|---|
| 62 Unique Paths | `up + left` |
| 63 Unique Paths II | obstacle → 0, else `up + left` |
| 64 Minimum Path Sum | `grid[r][c] + min(up, left)` |
| 221 Maximal Square | `1 + min(up, left, diag)` when the cell is 1 |
| 120 Triangle | bottom-up: `row[j] + min(below[j], below[j + 1])` |
| 329 Longest Increasing Path | off-template: no fixed order → memoised DFS from every cell |

In [48]:
def min_path_sum(grid):
    """Template: rolling row; dp[c] holds 'up', dp[c - 1] holds 'left'."""
    cols = len(grid[0])
    dp = [float("inf")] * cols
    dp[0] = 0
    for row in grid:
        for c in range(cols):
            left = dp[c - 1] if c else float("inf")
            # CHANGE: the combine line
            dp[c] = row[c] + min(dp[c], left)
    return dp[-1]


def longest_increasing_path(matrix):
    """329: memoised DFS - order comes from the values, not the grid."""
    rows, cols = len(matrix), len(matrix[0])

    @cache
    def dfs(r, c):
        best = 1
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and matrix[nr][nc] > matrix[r][c]:
                best = max(best, 1 + dfs(nr, nc))
        return best

    return max(dfs(r, c) for r in range(rows) for c in range(cols))


print(
    min_path_sum([[1, 3, 1], [1, 5, 1], [4, 2, 1]]),
    longest_increasing_path([[9, 9, 4], [6, 6, 8], [2, 1, 1]]),
)

7 4


### 4.5.8 Palindrome DP · 🟡

**Practice:** 06 §5.1 · **Template:** expand around centre · interval DP

The condition: **substring (contiguous) or subsequence?**

| Question | Template |
|---|---|
| 5 Longest Palindromic Substring | expand from all `2n − 1` centres, keep the longest |
| 647 Palindromic Substrings | expand, count every step that still matches |
| 516 Longest Palindromic Subsequence | `dp[i][j]`: ends match → `2 + dp[i+1][j-1]`, else `max(dp[i+1][j], dp[i][j-1])` |

In [49]:
def expand(s, lo, hi):
    """Grow while both ends match; return the final (lo, hi) exclusive bounds."""
    while lo >= 0 and hi < len(s) and s[lo] == s[hi]:
        lo, hi = lo - 1, hi + 1
    return lo + 1, hi


def longest_palindrome(s):
    """5: try every odd and even centre."""
    best = (0, 0)
    for centre in range(len(s)):
        for lo, hi in (expand(s, centre, centre), expand(s, centre, centre + 1)):
            if hi - lo > best[1] - best[0]:
                best = (lo, hi)
    return s[best[0] : best[1]]


def longest_pal_subseq(s):
    """516: interval DP, shorter intervals first."""
    n = len(s)
    dp = [[0] * n for _ in range(n)]
    for i in range(n - 1, -1, -1):
        dp[i][i] = 1
        for j in range(i + 1, n):
            if s[i] == s[j]:
                dp[i][j] = 2 + dp[i + 1][j - 1]
            else:
                dp[i][j] = max(dp[i + 1][j], dp[i][j - 1])
    return dp[0][-1]


print(longest_palindrome("babad"), longest_pal_subseq("bbbab"))

bab 4


### 4.5.9 Two-string alignment DP · 🟢

**Practice:** 06 §5.2 · **Template:** `dp[i][j]` over prefixes `a[:i]`, `b[:j]`

| Question | Match | Mismatch | Base |
|---|---|---|---|
| 1143 Longest Common Subsequence | `diag + 1` | `max(up, left)` | 0 |
| 72 Edit Distance | `diag` | `1 + min(up, left, diag)` | `i` / `j` |
| 97 Interleaving String | `dp[i][j] = (up and a[i-1] == c[i+j-1]) or (left and b[j-1] == c[i+j-1])` | | `dp[0][0] = True` |

In [50]:
def align(a, b, base, match, mismatch):
    """Template: dp[i][j] = answer for a[:i] vs b[:j]."""
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(len(a) + 1):
        for j in range(len(b) + 1):
            if i == 0 or j == 0:
                # CHANGE: base row / column
                dp[i][j] = base(i, j)
            elif a[i - 1] == b[j - 1]:
                # CHANGE: the match line
                dp[i][j] = match(dp, i, j)
            else:
                # CHANGE: the mismatch line
                dp[i][j] = mismatch(dp, i, j)
    return dp[-1][-1]


# 1143 LCS
print(
    align(
        "abcde",
        "ace",
        lambda i, j: 0,
        lambda d, i, j: d[i - 1][j - 1] + 1,
        lambda d, i, j: max(d[i - 1][j], d[i][j - 1]),
    )
)
# 72 Edit distance
print(
    align(
        "horse",
        "ros",
        lambda i, j: i + j,
        lambda d, i, j: d[i - 1][j - 1],
        lambda d, i, j: 1 + min(d[i - 1][j], d[i][j - 1], d[i - 1][j - 1]),
    )
)

3
3


### 4.5.10 Interval and game DP · 🟢

**Practice:** 06 §6.1 · **Template:** `dp(i, j)` = best score lead for the player to move

| Question | What you change |
|---|---|
| 486 Predict the Winner | return `dp(0, n - 1) >= 0` |
| 877 Stone Game | same (and the first player always wins with an even pile count) |

`dp(i, j) = max(nums[i] - dp(i + 1, j), nums[j] - dp(i, j - 1))`: take an end, then the opponent's lead is subtracted.

In [51]:
def first_player_lead(nums):
    """Template: the score lead the player to move can force on nums[i..j]."""

    @cache
    def dp(i, j):
        if i > j:
            return 0
        return max(nums[i] - dp(i + 1, j), nums[j] - dp(i, j - 1))

    return dp(0, len(nums) - 1)


print(first_player_lead([1, 5, 2]) >= 0, first_player_lead([1, 5, 233, 7]) >= 0)

False True


## 4.6 Greedy, Intervals, Math and Design

### 4.6.1 Greedy reachability and scans · 🔴

**Practice:** 07 §1.1 · **Template:** none — a handful of one-pass ideas

Each greedy question has its own "obvious once seen" rule. Learn the ideas, not a loop.

| Idea | Questions |
|---|---|
| Track the farthest reachable index (and where the current jump ends) | 55, 45 |
| Running total; when it goes negative, restart after this point | 134 (also 53) |
| Cut when you reach the last occurrence of every letter seen so far | 763 |
| Keep a range `[low, high]` of possible values | 678 |
| Two passes: left → right, then right → left | 135 |
| Sort by a clever key, then insert / take smallest first | 406, 846 |
| Keep the two smallest seen so far | 334 |
| Queues of indices that take turns | 649 |
| Element-wise max of the triplets that never overshoot | 1899 |

In [52]:
def min_jumps(nums):
    """55 / 45: farthest reach; a new jump when you pass the current end."""
    jumps = end = farthest = 0
    for i in range(len(nums) - 1):
        farthest = max(farthest, i + nums[i])
        # 55 check: if farthest <= i: unreachable
        if i == end:
            jumps, end = jumps + 1, farthest
    return jumps


def gas_start(gas, cost):
    """134: restart after any point where the tank goes negative."""
    if sum(gas) < sum(cost):
        return -1
    start = tank = 0
    for i in range(len(gas)):
        tank += gas[i] - cost[i]
        if tank < 0:
            start, tank = i + 1, 0
    return start


def partition_labels(s):
    """763: cut when you reach the last occurrence of everything inside."""
    last = {ch: i for i, ch in enumerate(s)}
    sizes, start, end = [], 0, 0
    for i, ch in enumerate(s):
        end = max(end, last[ch])
        if i == end:
            sizes.append(end - start + 1)
            start = i + 1
    return sizes


def candy(ratings):
    """135: two passes, each fixes one neighbour direction."""
    n = len(ratings)
    give = [1] * n
    for i in range(1, n):
        if ratings[i] > ratings[i - 1]:
            give[i] = give[i - 1] + 1
    for i in range(n - 2, -1, -1):
        if ratings[i] > ratings[i + 1]:
            give[i] = max(give[i], give[i + 1] + 1)
    return sum(give)


print(min_jumps([2, 3, 1, 1, 4]), gas_start([1, 2, 3, 4, 5], [3, 4, 5, 1, 2]))
print(partition_labels("ababcbacadefegdehijhklij"), candy([1, 0, 2]))

2 3
[9, 7, 8] 5


### 4.6.2 Intervals: merging and scheduling · 🟡

**Practice:** 07 §2.1 · **Template:** sort by start (merge) · sort by end (pick) · heap or sweep (overlap)

The condition: **are you merging, choosing, or counting overlaps?**

| Question | Sort by | What you change |
|---|---|---|
| 56 Merge Intervals | start | overlap → extend the last merged end |
| 57 Insert Interval | (already sorted) | before / overlapping / after the new one |
| 252 Meeting Rooms | start | any `start < previous end` → `False` |
| 435 Non-overlapping Intervals | **end** | keep if `start >= last_end`; removed = n − kept |
| 452 Min Arrows | **end** | new arrow if `start > arrow` |
| 253 Meeting Rooms II | start | min-heap of end times; pop if free |
| 1094 Car Pooling · 253 (alternative) | events | sweep: `(time, +1)` / `(time, −1)`, sort, running count; the peak is the answer |
| 986 Interval List Intersections | — | two pointers; overlap = `(max starts, min ends)`; advance the smaller end |
| 729 My Calendar I | — | sorted list + `bisect` to check both neighbours |

In [53]:
def merge(intervals):
    """Sort by start; extend or append."""
    out = []
    for start, end in sorted(intervals):
        # CHANGE: what an overlap does
        if out and start <= out[-1][1]:
            out[-1][1] = max(out[-1][1], end)
        else:
            out.append([start, end])
    return out


def max_non_overlapping(intervals):
    """435 / 452: sort by END; keep each interval starting after the last kept end."""
    kept, last_end = 0, float("-inf")
    for start, end in sorted(intervals, key=lambda iv: iv[1]):
        # CHANGE: >= for 435 (touching is fine), > for 452 arrows
        if start >= last_end:
            kept, last_end = kept + 1, end
    return kept


def min_rooms(intervals):
    """253: heap of end times = rooms in use."""
    ends = []
    for start, end in sorted(intervals):
        if ends and ends[0] <= start:
            heapq.heappop(ends)
        heapq.heappush(ends, end)
    return len(ends)


def intersections(a, b):
    """986: two pointers over two sorted lists."""
    i = j = 0
    out = []
    while i < len(a) and j < len(b):
        lo, hi = max(a[i][0], b[j][0]), min(a[i][1], b[j][1])
        if lo <= hi:
            out.append([lo, hi])
        if a[i][1] < b[j][1]:
            i += 1
        else:
            j += 1
    return out


def max_overlap(intervals):
    """Sweep line: ends sort before starts at the same time (-1 < +1)."""
    events = []
    for start, end in intervals:
        events += [(start, 1), (end, -1)]
    running = peak = 0
    for _, delta in sorted(events):
        running += delta
        peak = max(peak, running)
    return peak


ivs = [[1, 3], [2, 6], [8, 10], [15, 18]]
print(
    merge(ivs),
    4 - max_non_overlapping([[1, 2], [2, 3], [3, 4], [1, 3]]),
    min_rooms([[0, 30], [5, 10], [15, 20]]),
)
print(
    intersections([[0, 2], [5, 10]], [[1, 5], [8, 12]]),
    max_overlap([[0, 30], [5, 10], [15, 20]]),
)

[[1, 6], [8, 10], [15, 18]] 1 2
[[1, 2], [5, 5], [8, 10]] 2


### 4.6.3 Number manipulation · 🔴

**Practice:** 07 §3.1 · **Template:** none — digit loop, carry walk, fast power

| Idea | Questions |
|---|---|
| Digit loop: `n % 10`, `n //= 10` | 7, 202 |
| Seen-set to detect a cycle | 202 |
| Carry walk from the right | 66, 43 (and 2 Add Two Numbers) |
| Power by squaring: halve `n`, square `x` | 50 |

In [54]:
def digits(n):
    """Digit loop (least significant first)."""
    out = []
    while n:
        n, d = divmod(n, 10)
        out.append(d)
    return out


def plus_one(nums):
    """66: carry walk from the right."""
    nums = nums.copy()
    for i in range(len(nums) - 1, -1, -1):
        if nums[i] < 9:
            nums[i] += 1
            return nums
        nums[i] = 0
    return [1] + nums


def fast_pow(x, n):
    """50: square the base, halve the exponent."""
    if n < 0:
        x, n = 1 / x, -n
    result = 1.0
    while n:
        if n & 1:
            result *= x
        x *= x
        n >>= 1
    return result


def multiply(a, b):
    """43: grade-school product; a[i] * b[j] lands at slot i + j + 1."""
    out = [0] * (len(a) + len(b))
    for i in range(len(a) - 1, -1, -1):
        for j in range(len(b) - 1, -1, -1):
            total = out[i + j + 1] + int(a[i]) * int(b[j])
            out[i + j + 1] = total % 10
            out[i + j] += total // 10
    return "".join(map(str, out)).lstrip("0") or "0"


print(digits(123), plus_one([9, 9]), fast_pow(2.0, 10), multiply("123", "456"))

[3, 2, 1] [1, 0, 0] 1024.0 56088


### 4.6.4 Geometry and randomness · 🔴

**Practice:** 07 §3.2 · **Template:** none — three separate ideas

| Question | Idea |
|---|---|
| 384 Shuffle an Array | Fisher–Yates: swap `i` with a random index in `[i, n)` |
| 2013 Detect Squares | Counter of points; for each diagonal partner, multiply the counts of the two other corners |
| 382 Linked List Random Node | reservoir sampling: the i-th item replaces the pick with probability 1/i (k items: replace slot `j` if `randrange(i) < k`) |

In [55]:
def fisher_yates(nums, rng):
    """384: every permutation equally likely."""
    nums = nums.copy()
    for i in range(len(nums)):
        j = rng.randrange(i, len(nums))
        nums[i], nums[j] = nums[j], nums[i]
    return nums


def count_squares(points, query):
    """2013: pick the diagonal corner, the other two are then fixed."""
    count, (qx, qy) = Counter(map(tuple, points)), query
    total = 0
    for (x, y), c in count.items():
        if abs(x - qx) == abs(y - qy) and x != qx:
            total += c * count[(qx, y)] * count[(x, qy)]
    return total


def reservoir(stream, k, rng):
    """k uniform picks from a stream of unknown length, O(k) memory."""
    picks = []
    for i, x in enumerate(stream, 1):
        if i <= k:
            picks.append(x)
        else:
            # keep x with probability k / i, evicting a random slot
            j = rng.randrange(i)
            if j < k:
                picks[j] = x
    return picks


rng = random.Random(0)
hits = Counter(x for _ in range(3000) for x in reservoir(range(6), 2, rng))
print(reservoir(range(100), 3, rng), dict(sorted(hits.items())))
print(
    fisher_yates([1, 2, 3, 4], random.Random(0)),
    count_squares([[3, 10], [11, 2], [3, 2], [11, 2]], [11, 10]),
)

[39, 56, 7] {0: 967, 1: 941, 2: 1025, 3: 1074, 4: 975, 5: 1018}
[4, 3, 2, 1] 2


### 4.6.5 Bit manipulation · 🔴

**Practice:** 07 §4.1 · **Template:** none — a trick sheet

| Trick | Questions |
|---|---|
| `x ^ x = 0`, `x ^ 0 = x` (pairs cancel) | 136, 268 |
| `n & (n - 1)` drops the lowest 1-bit | 191 |
| `bits[i] = bits[i >> 1] + (i & 1)` | 338 |
| 32-step loop: read bit `n & 1`, shift | 190 |
| Add without `+`: sum = `a ^ b`, carry = `(a & b) << 1`, mask to 32 bits | 371 |
| AND of a range = common binary prefix (shift both until equal) | 201 |

In [56]:
def single_number(nums):
    """136: pairs cancel under XOR."""
    out = 0
    for x in nums:
        out ^= x
    return out


def popcount(n):
    """191: each step removes one 1-bit."""
    count = 0
    while n:
        n &= n - 1
        count += 1
    return count


def counting_bits(n):
    """338: reuse the answer for i >> 1."""
    bits = [0] * (n + 1)
    for i in range(1, n + 1):
        bits[i] = bits[i >> 1] + (i & 1)
    return bits


def reverse_bits(n):
    """190: read from the right, write from the left."""
    out = 0
    for _ in range(32):
        out = (out << 1) | (n & 1)
        n >>= 1
    return out


def range_and(lo, hi):
    """201: shift until the numbers agree, then shift back."""
    shift = 0
    while lo != hi:
        lo, hi, shift = lo >> 1, hi >> 1, shift + 1
    return lo << shift


print(
    single_number([4, 1, 2, 1, 2]),
    popcount(11),
    counting_bits(5),
    reverse_bits(1) == 2**31,
    range_and(5, 7),
)

4 3 [0, 1, 1, 2, 1, 2] True 4


### 4.6.6 Data-structure design · 🔴

**Practice:** 07 §5.1 · **Template:** none — pick one structure per operation

No shared code, but a shared recipe: **list every operation with its required cost, pick a structure for each, keep them in sync.**

| Question | Structures |
|---|---|
| 380 Insert Delete GetRandom O(1) | list + value → index map; delete = swap with the last, pop |
| 362 Design Hit Counter | deque of timestamps; drop those older than 300 s |
| 622 Design Circular Queue | fixed array + head index + count (mod arithmetic) |
| 284 Peeking Iterator | cache the next element |
| 341 Flatten Nested List Iterator | stack of reversed lists, flattened lazily in `has_next` |
| 1396 Underground System | check-ins map + `(from, to) → [total, count]` |

In [57]:
class RandomizedSet:
    """380: list for random choice, map for O(1) find."""

    def __init__(self, rng):
        self.items, self.where, self.rng = [], {}, rng

    def insert(self, x):
        if x in self.where:
            return False
        self.where[x] = len(self.items)
        self.items.append(x)
        return True

    def remove(self, x):
        if x not in self.where:
            return False
        # move the last item into x's slot, then pop the end
        i, last = self.where.pop(x), self.items.pop()
        if i < len(self.items):
            self.items[i], self.where[last] = last, i
        return True

    def get_random(self):
        return self.rng.choice(self.items)


class CircularQueue:
    """622: head + count over a fixed array."""

    def __init__(self, k):
        self.data, self.head, self.count = [0] * k, 0, 0

    def enqueue(self, x):
        if self.count == len(self.data):
            return False
        self.data[(self.head + self.count) % len(self.data)] = x
        self.count += 1
        return True

    def dequeue(self):
        if not self.count:
            return False
        self.head, self.count = (self.head + 1) % len(self.data), self.count - 1
        return True

    def front(self):
        return self.data[self.head] if self.count else -1


rs = RandomizedSet(random.Random(0))
print(rs.insert(1), rs.insert(2), rs.remove(1), rs.get_random())
q = CircularQueue(2)
print(q.enqueue(1), q.enqueue(2), q.enqueue(3), q.dequeue(), q.front())

True True True 2
True True False True 2
